# 🚀 Uplift Pilot — Model Benchmarking

Полное сравнение подходов с OOF-валидацией на `uplift@10`.

In [59]:
%%bash
pip install lightgbm causalml econml scikit-uplift optuna shap pandas pyarrow scikit-learn numpy scipy -q


[notice] A new release of pip is available: 26.0.1 -> 26.1.1
[notice] To update, run: python3.14 -m pip install --upgrade pip


In [9]:
# ══════════════════════════════════════════════════════════════════════════════
# MODEL 15: Seed-Averaged Blend (per-channel × 0.7 + global × 0.3)
# ══════════════════════════════════════════════════════════════════════════════
from scipy.stats import rankdata

SEEDS = [42, 7, 13, 21, 99]

def train_xlearner_score_test(params1, params2, prop, X_tr, y_tr, T_tr, X_te, seed):
    m1 = T_tr == 1
    m0 = T_tr == 0

    mu1 = lgb.LGBMRegressor(**{**params1, 'random_state': seed})
    mu0 = lgb.LGBMRegressor(**{**params1, 'random_state': seed})
    mu1.fit(X_tr[m1], y_tr[m1])
    mu0.fit(X_tr[m0], y_tr[m0])

    D1 = y_tr[m1] - mu0.predict(X_tr[m1])
    D0 = mu1.predict(X_tr[m0]) - y_tr[m0]

    tau1 = lgb.LGBMRegressor(**{**params2, 'random_state': seed})
    tau0 = lgb.LGBMRegressor(**{**params2, 'random_state': seed})
    tau1.fit(X_tr[m1], D1)
    tau0.fit(X_tr[m0], D0)

    if prop == 'logreg':
        g = LogisticRegression(max_iter=500, C=1.0, n_jobs=-1)
    else:
        g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                n_jobs=-1, random_state=seed, verbose=-1)
    g.fit(X_tr, T_tr)
    g_te = g.predict_proba(X_te)[:, 1]

    return g_te * tau0.predict(X_te) + (1 - g_te) * tau1.predict(X_te)


# ── Загружаем параметры глобальной модели ─────────────────────────────────────
with open(f'{ARTIFACTS_DIR}/optuna_xl_params.json') as f:
    _p = json.load(f)

GLOBAL_PARAMS1 = {**_p['params1'], 'n_jobs': -1, 'verbose': -1}
GLOBAL_PARAMS2 = {**_p['params2'], 'n_jobs': -1, 'verbose': -1}
GLOBAL_PROP    = _p['prop_model']
GLOBAL_FEAT    = FEAT_COLS if _p['feat_set'] == 'all' else FEAT_COLS_TOP

print(f'Global model: prop={GLOBAL_PROP}  feat_set={_p["feat_set"]}')

# ── Загружаем per-channel параметры ───────────────────────────────────────────
perchan_path = f'{ARTIFACTS_DIR}/perchan_xl_params.json'
if os.path.exists(perchan_path):
    with open(perchan_path) as f:
        _raw = json.load(f)
    PERCHAN_PARAMS = {
        ch_str: {
            'params1': {**d['params1'], 'n_jobs': -1, 'verbose': -1},
            'params2': {**d['params2'], 'n_jobs': -1, 'verbose': -1},
            'prop':    d['prop'],
            'feat':    FEAT_COLS if d.get('feat', 'all') == 'all' else FEAT_COLS_TOP,
        }
        for ch_str, d in _raw.items()
    }
    print(f'Per-channel params loaded: {list(PERCHAN_PARAMS.keys())}')
else:
    print('WARNING: perchan_xl_params.json not found — using global params for all channels')
    PERCHAN_PARAMS = {
        str(ch): {'params1': GLOBAL_PARAMS1, 'params2': GLOBAL_PARAMS2,
                  'prop': GLOBAL_PROP, 'feat': GLOBAL_FEAT}
        for ch in sorted(train[COMM_COL].unique())
    }

comm_train = train[COMM_COL].values
comm_test  = test[COMM_COL].values
COMM_CODES = sorted(train[COMM_COL].unique())


# ══════════════════════════════════════════════════════════════════════════════
# Часть 1: Глобальная модель — seed averaging
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Part 1: Global X-Learner seed averaging')
t0 = time.time()

X_tr_global = encode_X(train, GLOBAL_FEAT).values
X_te_global = encode_X(test,  GLOBAL_FEAT).values

global_rank_sum = np.zeros(len(test))
for i, seed in enumerate(SEEDS):
    print(f'  seed={seed} ({i+1}/{len(SEEDS)})...', end=' ', flush=True)
    t_s = time.time()
    sc = train_xlearner_score_test(
        GLOBAL_PARAMS1, GLOBAL_PARAMS2, GLOBAL_PROP,
        X_tr_global, y, T,
        X_te_global, seed
    )
    global_rank_sum += rankdata(sc)
    print(f'{time.time()-t_s:.0f}s')

test_scores_global_avg = global_rank_sum / len(SEEDS)
print(f'Global done in {time.time()-t0:.0f}s')
np.save(f'{ARTIFACTS_DIR}/test_scores_global_seed_avg.npy', test_scores_global_avg)


# ══════════════════════════════════════════════════════════════════════════════
# Часть 2: Per-channel модель — seed averaging
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
print('Part 2: Per-Channel X-Learner seed averaging')
t0 = time.time()

perchan_rank_sum = np.zeros(len(test))
for i, seed in enumerate(SEEDS):
    print(f'\n  Seed {seed} ({i+1}/{len(SEEDS)}):')
    test_scores_this_seed = np.zeros(len(test))

    for ch in COMM_CODES:
        ch_str    = str(ch)
        cp        = PERCHAN_PARAMS.get(ch_str, list(PERCHAN_PARAMS.values())[0])
        feat_cols = cp['feat']

        ch_tr_mask = comm_train == ch
        ch_te_mask = comm_test  == ch

        X_tr_ch = encode_X(train[ch_tr_mask].reset_index(drop=True), feat_cols).values
        X_te_ch = encode_X(test[ch_te_mask].reset_index(drop=True),  feat_cols).values
        y_ch    = y[ch_tr_mask]
        T_ch    = T[ch_tr_mask]

        print(f'    ch={ch} n_tr={ch_tr_mask.sum():,} n_te={ch_te_mask.sum():,}...',
              end=' ', flush=True)
        t_ch = time.time()

        sc_ch = train_xlearner_score_test(
            cp['params1'], cp['params2'], cp['prop'],
            X_tr_ch, y_ch, T_ch,
            X_te_ch, seed
        )
        test_scores_this_seed[ch_te_mask] = sc_ch
        print(f'{time.time()-t_ch:.0f}s')

    perchan_rank_sum += rankdata(test_scores_this_seed)

test_scores_perchan_avg = perchan_rank_sum / len(SEEDS)
print(f'\nPer-channel done in {time.time()-t0:.0f}s')
np.save(f'{ARTIFACTS_DIR}/test_scores_perchan_seed_avg.npy', test_scores_perchan_avg)


# ══════════════════════════════════════════════════════════════════════════════
# Часть 3: Финальный бленд 0.7 × perchan + 0.3 × global
# ══════════════════════════════════════════════════════════════════════════════
print('\n' + '='*60)
blend_final = 0.7 * test_scores_perchan_avg + 0.3 * test_scores_global_avg

n_top    = int(0.1 * len(test))
top_mask = blend_final >= np.sort(blend_final)[::-1][n_top]
print(f'Состав топ-10% теста ({n_top:,} клиентов):')
for ch in COMM_CODES:
    n_ch = ((comm_test == ch) & top_mask).sum()
    print(f'  ch={ch}: {n_ch:,} ({n_ch/n_top*100:.1f}%)')

submission = pd.DataFrame({
    'user_id':      test[ID_COL],
    'UPLIFT_SCORE': blend_final,
})
sub_path = f'{ARTIFACTS_DIR}/submission_blend_perchan07_global03_seedavg.csv'
submission.to_csv(sub_path, index=False)

print(f'\n✅ Saved: {sub_path}')
print(f'   shape={submission.shape}')
print(f'   score stats: min={blend_final.min():.1f}  max={blend_final.max():.1f}  '
      f'mean={blend_final.mean():.1f}  std={blend_final.std():.1f}')
print(submission.head())

Global model: prop=logreg  feat_set=all
Per-channel params loaded: ['0', '1', '2']

Part 1: Global X-Learner seed averaging


KeyError: "['communicationtype', 'gender_calc', 'p25tv', 'p75tv', 'rto_format1', 'rto_format2', 'rto_format3', 'n_cat7', 'n_cat6', 'n_cat5', 'p25_days_btw_trn', 'p50_days_btw_trn', 'p75_days_btw_trn', 'cus_cat7_rto', 'cus_cat7_atv', 'cus_cat7_std', 'cus_cat7_n_days', 'cus_cat7_n_sku', 'cus_cat6_rto', 'cus_cat6_atv', 'cus_cat6_std', 'cus_cat6_n_days', 'cus_cat6_n_sku', 'cus_cat5_rto', 'cus_cat5_atv', 'cus_cat5_std', 'cus_cat5_n_days', 'cus_cat5_n_sku', 'cus_cat7_n_days_big_period', 'cus_catdays7_diff_mean', 'cus_catdays7_diff_min', 'cus_catdays7_diff_max', 'cus_catdays7_diff_std', 'cus_cat7_last1_2days', 'cus_cat7_last2_3days', 'cus_cat7_max_min_days_diff', 'cus_cat7_last1days', 'cus_cat7_last2days', 'cus_cat7_last3days', 'cus_cat6_n_days_big_period', 'cus_catdays6_diff_mean', 'cus_catdays6_diff_min', 'cus_catdays6_diff_max', 'cus_catdays6_diff_std', 'cus_cat6_last1_2days', 'cus_cat6_last2_3days', 'cus_cat6_max_min_days_diff', 'cus_cat6_last1days', 'cus_cat6_last2days', 'cus_cat6_last3d

## 1. Config & Imports

In [1]:
import warnings, json, time, os
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
from scipy.stats import mannwhitneyu

import lightgbm as lgb
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import StratifiedKFold
from sklearn.linear_model import LogisticRegression

from sklift.metrics import uplift_at_k, qini_auc_score
from sklift.metrics import uplift_curve, qini_curve

import optuna
optuna.logging.set_verbosity(optuna.logging.WARNING)

import shap
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.ticker as mticker

# ── Constants ──────────────────────────────────────────────────────────────────
RANDOM_STATE   = 42
N_FOLDS        = 5
ARTIFACTS_DIR  = 'pilot_artifacts'
os.makedirs(ARTIFACTS_DIR, exist_ok=True)

TARGET_COL    = 'rec_spend'
TREATMENT_COL = 'treatment_flg'
ID_COL        = 'user_id'
COMM_COL      = 'communication_type'

COMM_TYPES    = ['com_type_1', 'com_type_2', 'com_type_3']

# TOP-25 из EDA feature importance
TOP25 = [
    'cus_cat_7_rto', 'n_redeem', 'cus_cat_7_atv', 'cus_cat_5_rto',
    'cus_cat_5_std', 'mntv', 'n_days_life', 'cus_cat_5_atv',
    'cus_cat_7_n_days_big_period', 'cus_cat_7_std', 'cus_mark_n_offers',
    'cus_cat_6_rto', 'rto_format_1', 'age', 'cus_mark_n_view', 'n_sku',
    'cus_cat_6_std', 'cus_cat_6_atv', 'cus_cat_5_max_min_days_diff',
    'cus_mark_n_rule', 'p_25_tv', 'cus_cat_7_max_min_days_diff',
    'mxtv', 'rto_format_3', 'avg_days_btw_trn'
]

print('✅ Config loaded')
print(f'Artifacts dir: {ARTIFACTS_DIR}/')

✅ Config loaded
Artifacts dir: pilot_artifacts/


## 2. Load & Prepare Data

In [2]:
train = pd.read_parquet('train.parquet')
test  = pd.read_parquet('test.parquet')
print(f'Train: {train.shape}, Test: {test.shape}')
print(f'Treatment balance: T={train[TREATMENT_COL].mean():.3f}')

ALL_FEATS = [c for c in train.columns
             if c not in [ID_COL, TARGET_COL, TREATMENT_COL]]

# Label-encode строковые колонки
le_map = {}
for col in train[ALL_FEATS].select_dtypes(include=['object','string','category']).columns:
    le = LabelEncoder()
    combined = pd.concat([train[col], test[col]], ignore_index=True).astype(str)
    le.fit(combined)
    train[col] = le.transform(train[col].astype(str))
    test[col]  = le.transform(test[col].astype(str))
    le_map[col] = le

print(f'Features: {len(ALL_FEATS)}')
print(f'Target zeros: {(train[TARGET_COL]==0).mean():.1%}')

Train: (355246, 89), Test: (118414, 87)
Treatment balance: T=0.496
Features: 86
Target zeros: 90.2%


## 3. Feature Engineering

Новые фичи на основе EDA: ratio к категории, маркетинговая отзывчивость, recency-ratio.

In [3]:
def engineer_features(df):
    df = df.copy()
    eps = 1e-6

    # Affinity to target category
    df['cat7_share']       = df['cus_cat_7_rto'] / (df['rto'] + eps)
    df['cat6_share']       = df['cus_cat_6_rto'] / (df['rto'] + eps)
    df['cat5_share']       = df['cus_cat_5_rto'] / (df['rto'] + eps)

    # Marketing responsiveness
    df['mkt_resp_rate']    = df['cus_mark_n_rule'] / (df['cus_mark_n_offers'].clip(lower=1))
    df['mkt_view_rate']    = df['cus_mark_n_view']  / (df['cus_mark_n_offers'].clip(lower=1))

    # Recency relative to avg purchase cadence
    df['recency_ratio']    = df['cus_cat_7_last_1_days'] / (df['avg_days_btw_trn'] + eps)

    # Spend variability
    df['spend_cv']         = df['stdtv'] / (df['atv'] + eps)

    # Days life × transaction frequency
    df['trn_density']      = df['n_trn'] / (df['n_days_life'] + eps) * 365

    # Category breadth in target hierarchy
    df['cat_breadth_ratio'] = df['n_cat_7'] / (df['n_cat_5'] + eps)

    # How fresh is the category purchase vs general last visit
    df['cat7_vs_last_visit'] = df['cus_cat_7_last_1_days'] / (df['n_days_last_visit'] + eps)

    return df

train = engineer_features(train)
test  = engineer_features(test)

NEW_FEATS = [
    'cat7_share','cat6_share','cat5_share',
    'mkt_resp_rate','mkt_view_rate',
    'recency_ratio','spend_cv','trn_density',
    'cat_breadth_ratio','cat7_vs_last_visit'
]

FEAT_COLS = ALL_FEATS + NEW_FEATS
FEAT_COLS_TOP = TOP25 + NEW_FEATS  # расширенный топ

print(f'Total features after engineering: {len(FEAT_COLS)}')
print(f'New features: {NEW_FEATS}')

Total features after engineering: 96
New features: ['cat7_share', 'cat6_share', 'cat5_share', 'mkt_resp_rate', 'mkt_view_rate', 'recency_ratio', 'spend_cv', 'trn_density', 'cat_breadth_ratio', 'cat7_vs_last_visit']


## 4. Validation Utilities

ОOF uplift@10 со стратификацией по `(treatment_flg × communication_type)`.

In [4]:
def make_stratify_col(df):
    """6-страт: treatment × communication_type"""
    return df[TREATMENT_COL].astype(str) + '_' + df[COMM_COL].astype(str)

def uplift_at_k_spend(y_spend, uplift_scores, treatment, k=0.1):
    """
    uplift@k по непрерывному rec_spend:
    E[rec_spend | T=1, top-k%] - E[rec_spend | T=0, top-k%]
    """
    n = len(y_spend)
    top_n = max(int(n * k), 1)
    
    # Берём индексы топ-k% по uplift score
    top_idx = np.argsort(uplift_scores)[::-1][:top_n]
    
    t_mask  = treatment[top_idx] == 1
    c_mask  = treatment[top_idx] == 0
    
    if t_mask.sum() == 0 or c_mask.sum() == 0:
        return 0.0
    
    return float(y_spend[top_idx][t_mask].mean() - y_spend[top_idx][c_mask].mean())


def oof_uplift_at_k(y_true, scores, treatment, k=0.1, n_boot=100, seed=42):
    """Возвращает (mean, ci80_lo, ci80_hi) по rec_spend"""
    point = uplift_at_k_spend(y_true, scores, treatment, k)
    rng   = np.random.default_rng(seed)
    idx   = np.arange(len(y_true))
    boots = []
    for _ in range(n_boot):
        s = rng.choice(idx, len(idx), replace=True)
        boots.append(uplift_at_k_spend(y_true[s], scores[s], treatment[s], k))
    boots = np.array(boots)
    return float(point), float(np.percentile(boots, 10)), float(np.percentile(boots, 90))

def oof_qini(y_true, scores, treatment):
    y_bin = (y_true > 0).astype(int)
    return float(qini_auc_score(y_bin, scores, treatment))

def get_folds(df, n_splits=N_FOLDS):
    strat = make_stratify_col(df)
    skf = StratifiedKFold(n_splits=n_splits, shuffle=True, random_state=RANDOM_STATE)
    return list(skf.split(df, strat))

RESULTS = {}  # dict: model_name -> metrics

def register_result(name, y_val, scores_val, treat_val, elapsed):
    u10, ci_lo, ci_hi = oof_uplift_at_k(y_val, scores_val, treat_val)
    qauc = oof_qini(y_val, scores_val, treat_val)
    
    entry = {
        'model':       name,
        'uplift@10':   u10,
        'ci80_lo':     ci_lo,
        'ci80_hi':     ci_hi,
        'qini_auc':    qauc,
        'elapsed_sec': round(elapsed, 1)
    }
    
    # Обновляем если уже есть, иначе добавляем
    for i, r in enumerate(RESULTS):
        if r['model'] == name:
            RESULTS[i] = entry
            break
    else:
        RESULTS.append(entry)
    
    print(f'  uplift@10={u10:.4f}  CI80=[{ci_lo:.4f}, {ci_hi:.4f}]  '
          f'qini={qauc:.4f}  ({elapsed:.0f}s)')
    return u10

print('✅ Validation utils ready')

✅ Validation utils ready


## 5. Base LGB Parameters

In [5]:
BASE_LGB_REG = dict(
    n_estimators=500, learning_rate=0.05, num_leaves=127,
    min_child_samples=20, subsample=0.8, colsample_bytree=0.8,
    reg_alpha=0.1, reg_lambda=1.0,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
BASE_LGB_CLF = {**BASE_LGB_REG}

def encode_X(df, feat_cols):
    X = df[feat_cols].copy().fillna(-999)
    for c in X.select_dtypes(include=['object','string','category']).columns:
        X[c] = X[c].astype('category')
    return X

print('✅ LGB params ready')

✅ LGB params ready


## 6. Model 1 — T-Learner (Baseline)

In [8]:
print('Training Model 1: T-Learner...')
t0 = time.time()

folds = get_folds(train)
oof_scores_tl = np.zeros(len(train))

y  = train[TARGET_COL].values
T  = train[TREATMENT_COL].values

# for fold, (tr_idx, val_idx) in enumerate(folds):
#     X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
#     X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
#     y_tr  = y[tr_idx]; T_tr = T[tr_idx]

#     m1 = lgb.LGBMRegressor(**BASE_LGB_REG)
#     m0 = lgb.LGBMRegressor(**BASE_LGB_REG)
#     m1.fit(X_tr[T_tr==1], y_tr[T_tr==1])
#     m0.fit(X_tr[T_tr==0], y_tr[T_tr==0])

#     oof_scores_tl[val_idx] = m1.predict(X_val) - m0.predict(X_val)

# elapsed_tl = time.time() - t0
# u10_tl = register_result('T-Learner', y, oof_scores_tl, T, elapsed_tl)
# np.save(f'{ARTIFACTS_DIR}/oof_tlearner.npy', oof_scores_tl)

Training Model 1: T-Learner...


## 7. Model 2 — Hurdle T-Learner

Stage 1: P(buy>0), Stage 2: E[spend|buy>0, log-scale]. Uplift = P1·E1 − P0·E0.

In [ ]:
print('Training Model 2: Hurdle T-Learner...')
t0 = time.time()

oof_scores_hurdle = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
    X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
    y_tr  = y[tr_idx]; T_tr = T[tr_idx]
    y_bin_tr = (y_tr > 0).astype(int)

    # Stage 1: P(buy)
    clf1 = lgb.LGBMClassifier(**BASE_LGB_CLF)
    clf0 = lgb.LGBMClassifier(**BASE_LGB_CLF)
    clf1.fit(X_tr[T_tr==1], y_bin_tr[T_tr==1])
    clf0.fit(X_tr[T_tr==0], y_bin_tr[T_tr==0])
    p1 = clf1.predict_proba(X_val)[:,1]
    p0 = clf0.predict_proba(X_val)[:,1]

    # Stage 2: E[spend | buy>0], log-scale
    mask_tr1 = (T_tr==1) & (y_tr > 0)
    mask_tr0 = (T_tr==0) & (y_tr > 0)
    reg1 = lgb.LGBMRegressor(**BASE_LGB_REG)
    reg0 = lgb.LGBMRegressor(**BASE_LGB_REG)
    reg1.fit(X_tr[mask_tr1], np.log1p(y_tr[mask_tr1]))
    reg0.fit(X_tr[mask_tr0], np.log1p(y_tr[mask_tr0]))
    e1 = np.expm1(reg1.predict(X_val))
    e0 = np.expm1(reg0.predict(X_val))

    oof_scores_hurdle[val_idx] = p1 * e1 - p0 * e0

elapsed_hurdle = time.time() - t0
u10_hurdle = register_result('Hurdle T-Learner', y, oof_scores_hurdle, T, elapsed_hurdle)
np.save(f'{ARTIFACTS_DIR}/oof_hurdle.npy', oof_scores_hurdle)

Training Model 2: Hurdle T-Learner...
  [Hurdle T-Learner] uplift@10=0.0521 [0.0486, 0.0582]  Qini=0.0265  t=354s


## 8. Model 3 — X-Learner

Residualized CATE: τ₁ = Y(1)−Ŷ₁(X), τ₀ = Ŷ₀(X)−Y(0), финал взвешен propensity.

In [ ]:
print('Training Model 3: X-Learner...')
t0 = time.time()

oof_scores_xl = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
    X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
    y_tr  = y[tr_idx]; T_tr = T[tr_idx]

    # Step 1: T-learner outcome models
    m1 = lgb.LGBMRegressor(**BASE_LGB_REG)
    m0 = lgb.LGBMRegressor(**BASE_LGB_REG)
    m1.fit(X_tr[T_tr==1], y_tr[T_tr==1])
    m0.fit(X_tr[T_tr==0], y_tr[T_tr==0])

    # Step 2: residual CATE models
    tau1_tr = y_tr[T_tr==1] - m0.predict(X_tr[T_tr==1])  # imputed control outcome
    tau0_tr = m1.predict(X_tr[T_tr==0]) - y_tr[T_tr==0]  # imputed treatment outcome

    tau_m1 = lgb.LGBMRegressor(**BASE_LGB_REG)
    tau_m0 = lgb.LGBMRegressor(**BASE_LGB_REG)
    tau_m1.fit(X_tr[T_tr==1], tau1_tr)
    tau_m0.fit(X_tr[T_tr==0], tau0_tr)

    # Step 3: propensity weighting
    prop_clf = lgb.LGBMClassifier(**BASE_LGB_CLF)
    prop_clf.fit(X_tr, T_tr)
    e_val = prop_clf.predict_proba(X_val)[:,1].clip(0.05, 0.95)

    oof_scores_xl[val_idx] = e_val * tau_m0.predict(X_val) + (1-e_val) * tau_m1.predict(X_val)

elapsed_xl = time.time() - t0
u10_xl = register_result('X-Learner', y, oof_scores_xl, T, elapsed_xl)
np.save(f'{ARTIFACTS_DIR}/oof_xlearner.npy', oof_scores_xl)

Training Model 3: X-Learner...
  [X-Learner] uplift@10=0.0575 [0.0521, 0.0629]  Qini=0.0304  t=461s


## 9. Model 4 — DR-Learner (Doubly Robust)

Cross-fitting: nuisance models на I₁, pseudo-outcomes на I₂. Финальный регрессор на φ̂(Z).

In [ ]:
print('Training Model 4: DR-Learner...')
t0 = time.time()

oof_scores_dr = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
    X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
    y_tr  = y[tr_idx]; T_tr = T[tr_idx]

    # Cross-fitting: split train fold into two halves
    half = len(tr_idx) // 2
    idx_I1, idx_I2 = np.arange(half), np.arange(half, len(tr_idx))

    X_I1, X_I2 = X_tr.iloc[idx_I1], X_tr.iloc[idx_I2]
    y_I1, y_I2 = y_tr[idx_I1], y_tr[idx_I2]
    T_I1, T_I2 = T_tr[idx_I1], T_tr[idx_I2]

    def fit_nuisance(X_fit, y_fit, T_fit):
        m1_ = lgb.LGBMRegressor(**BASE_LGB_REG)
        m0_ = lgb.LGBMRegressor(**BASE_LGB_REG)
        ps_ = lgb.LGBMClassifier(**BASE_LGB_CLF)
        m1_.fit(X_fit[T_fit==1], y_fit[T_fit==1])
        m0_.fit(X_fit[T_fit==0], y_fit[T_fit==0])
        ps_.fit(X_fit, T_fit)
        return m1_, m0_, ps_

    # I1 → nuisance, compute pseudo-outcomes on I2
    m1_a, m0_a, ps_a = fit_nuisance(X_I1, y_I1, T_I1)
    mu1_I2 = m1_a.predict(X_I2)
    mu0_I2 = m0_a.predict(X_I2)
    pi_I2  = ps_a.predict_proba(X_I2)[:,1].clip(0.05,0.95)
    phi_I2 = (mu1_I2 - mu0_I2
              + T_I2*(y_I2 - mu1_I2)/pi_I2
              - (1-T_I2)*(y_I2 - mu0_I2)/(1-pi_I2))

    # I2 → nuisance, compute pseudo-outcomes on I1
    m1_b, m0_b, ps_b = fit_nuisance(X_I2, y_I2, T_I2)
    mu1_I1 = m1_b.predict(X_I1)
    mu0_I1 = m0_b.predict(X_I1)
    pi_I1  = ps_b.predict_proba(X_I1)[:,1].clip(0.05,0.95)
    phi_I1 = (mu1_I1 - mu0_I1
              + T_I1*(y_I1 - mu1_I1)/pi_I1
              - (1-T_I1)*(y_I1 - mu0_I1)/(1-pi_I1))

    # Stack and train final CATE model
    X_pseudo = pd.concat([X_I1, X_I2], ignore_index=True)
    phi_all  = np.concatenate([phi_I1, phi_I2])

    dr_final = lgb.LGBMRegressor(**BASE_LGB_REG)
    dr_final.fit(X_pseudo, phi_all)

    oof_scores_dr[val_idx] = dr_final.predict(X_val)

elapsed_dr = time.time() - t0
u10_dr = register_result('DR-Learner', y, oof_scores_dr, T, elapsed_dr)
np.save(f'{ARTIFACTS_DIR}/oof_drlearner.npy', oof_scores_dr)

Training Model 4: DR-Learner...
  [DR-Learner] uplift@10=0.0495 [0.0444, 0.0550]  Qini=0.0291  t=643s


## 10. Model 5 — Hurdle DR-Learner

ДR-learner применён к каждой стадии hurdle: ΔP(buy) и ΔE[spend|buy>0].

In [ ]:
print('Training Model 5: Hurdle DR-Learner...')
t0 = time.time()

oof_scores_hdr = np.zeros(len(train))

for fold, (tr_idx, val_idx) in enumerate(folds):
    X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
    X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
    y_tr  = y[tr_idx]; T_tr = T[tr_idx]
    y_bin_tr = (y_tr > 0).astype(int)
    y_log_tr = np.where(y_tr > 0, np.log1p(y_tr), np.nan)

    half = len(tr_idx) // 2
    idx_I1, idx_I2 = np.arange(half), np.arange(half, len(tr_idx))

    def dr_pseudo_binary(X_fit, y_fit, T_fit, X_pred):
        """DR pseudo-outcome for binary target (P(buy))"""
        m1_ = lgb.LGBMClassifier(**BASE_LGB_CLF)
        m0_ = lgb.LGBMClassifier(**BASE_LGB_CLF)
        ps_ = lgb.LGBMClassifier(**BASE_LGB_CLF)
        m1_.fit(X_fit[T_fit==1], y_fit[T_fit==1])
        m0_.fit(X_fit[T_fit==0], y_fit[T_fit==0])
        ps_.fit(X_fit, T_fit)
        mu1 = m1_.predict_proba(X_pred)[:,1]
        mu0 = m0_.predict_proba(X_pred)[:,1]
        pi  = ps_.predict_proba(X_pred)[:,1].clip(0.05,0.95)
        return mu1, mu0, pi

    def dr_pseudo_reg(X_fit, y_fit, T_fit, X_pred):
        """DR pseudo-outcome for continuous target"""
        m1_ = lgb.LGBMRegressor(**BASE_LGB_REG)
        m0_ = lgb.LGBMRegressor(**BASE_LGB_REG)
        ps_ = lgb.LGBMClassifier(**BASE_LGB_CLF)
        m1_.fit(X_fit[T_fit==1], y_fit[T_fit==1])
        m0_.fit(X_fit[T_fit==0], y_fit[T_fit==0])
        ps_.fit(X_fit, T_fit)
        mu1 = m1_.predict(X_pred)
        mu0 = m0_.predict(X_pred)
        pi  = ps_.predict_proba(X_pred)[:,1].clip(0.05,0.95)
        return mu1, mu0, pi

    # Stage 1: binary DR
    X_I1b, X_I2b = X_tr.iloc[idx_I1], X_tr.iloc[idx_I2]
    y_bin_I1, y_bin_I2 = y_bin_tr[idx_I1], y_bin_tr[idx_I2]
    T_I1, T_I2 = T_tr[idx_I1], T_tr[idx_I2]

    mu1_s1, mu0_s1, pi_s1 = dr_pseudo_binary(X_I1b, y_bin_I1, T_I1, X_I2b)
    phi_bin_I2 = (mu1_s1 - mu0_s1 + T_I2*(y_bin_I2-mu1_s1)/pi_s1
                  - (1-T_I2)*(y_bin_I2-mu0_s1)/(1-pi_s1))
    mu1_s1b, mu0_s1b, pi_s1b = dr_pseudo_binary(X_I2b, y_bin_I2, T_I2, X_I1b)
    phi_bin_I1 = (mu1_s1b - mu0_s1b + T_I1*(y_bin_I1-mu1_s1b)/pi_s1b
                  - (1-T_I1)*(y_bin_I1-mu0_s1b)/(1-pi_s1b))

    dr_bin = lgb.LGBMRegressor(**BASE_LGB_REG)
    dr_bin.fit(pd.concat([X_I1b,X_I2b],ignore_index=True),
               np.concatenate([phi_bin_I1, phi_bin_I2]))
    delta_p = dr_bin.predict(X_val)

    # Stage 2: reg DR on log-spend (только ненулевые)
    mask_pos_tr = y_tr > 0
    X_pos = X_tr[mask_pos_tr]
    y_pos = y_log_tr[mask_pos_tr]
    T_pos = T_tr[mask_pos_tr]
    if T_pos.sum() > 10 and (1-T_pos).sum() > 10:
        half_p = len(X_pos) // 2
        X_pI1, X_pI2 = X_pos.iloc[:half_p], X_pos.iloc[half_p:]
        y_pI1, y_pI2 = y_pos[:half_p], y_pos[half_p:]
        T_pI1, T_pI2 = T_pos[:half_p], T_pos[half_p:]

        mu1r, mu0r, pir = dr_pseudo_reg(X_pI1, y_pI1, T_pI1, X_pI2)
        phi_r_I2 = (mu1r - mu0r + T_pI2*(y_pI2-mu1r)/pir
                    - (1-T_pI2)*(y_pI2-mu0r)/(1-pir))
        mu1rb, mu0rb, pirb = dr_pseudo_reg(X_pI2, y_pI2, T_pI2, X_pI1)
        phi_r_I1 = (mu1rb - mu0rb + T_pI1*(y_pI1-mu1rb)/pirb
                    - (1-T_pI1)*(y_pI1-mu0rb)/(1-pirb))

        dr_reg = lgb.LGBMRegressor(**BASE_LGB_REG)
        dr_reg.fit(pd.concat([X_pI1,X_pI2],ignore_index=True),
                   np.concatenate([phi_r_I1, phi_r_I2]))
        delta_e = np.expm1(dr_reg.predict(X_val))
    else:
        delta_e = np.zeros(len(X_val))

    oof_scores_hdr[val_idx] = delta_p + delta_e * 0.5

elapsed_hdr = time.time() - t0
u10_hdr = register_result('Hurdle DR-Learner', y, oof_scores_hdr, T, elapsed_hdr)
np.save(f'{ARTIFACTS_DIR}/oof_hurdle_dr.npy', oof_scores_hdr)

Training Model 5: Hurdle DR-Learner...
  [Hurdle DR-Learner] uplift@10=0.0299 [0.0261, 0.0336]  Qini=0.0160  t=1217s


## 11. Model 6 — Per-Channel Hurdle T-Learner

EDA показал uplift 1.4 vs 3.8 vs 4.4 по каналам. Обучаем отдельные модели per `communication_type`.

In [ ]:
print('Training Model 6: Per-Channel Hurdle T-Learner...')
t0 = time.time()

oof_scores_perchan = np.zeros(len(train))
comm_val_enc = train[COMM_COL].values  # уже label-encoded

# Узнаём числовые коды каналов
comm_codes = sorted(train[COMM_COL].unique())

folds_perchan = get_folds(train)

for fold, (tr_idx, val_idx) in enumerate(folds_perchan):
    X_tr  = encode_X(train.iloc[tr_idx], FEAT_COLS)
    X_val = encode_X(train.iloc[val_idx], FEAT_COLS)
    y_tr  = y[tr_idx]; T_tr = T[tr_idx]
    comm_tr = comm_val_enc[tr_idx]
    comm_vl = comm_val_enc[val_idx]

    fold_pred = np.zeros(len(val_idx))

    for ch in comm_codes:
        mask_tr = comm_tr == ch
        mask_vl = comm_vl == ch

        if mask_tr.sum() < 100 or mask_vl.sum() == 0:
            continue

        Xch_tr = X_tr[mask_tr]; ych_tr = y_tr[mask_tr]; Tch_tr = T_tr[mask_tr]
        Xch_vl = X_val[mask_vl]

        # Hurdle per channel
        clf1 = lgb.LGBMClassifier(**BASE_LGB_CLF)
        clf0 = lgb.LGBMClassifier(**BASE_LGB_CLF)
        reg1 = lgb.LGBMRegressor(**BASE_LGB_REG)
        reg0 = lgb.LGBMRegressor(**BASE_LGB_REG)

        y_bin_ch = (ych_tr > 0).astype(int)
        clf1.fit(Xch_tr[Tch_tr==1], y_bin_ch[Tch_tr==1])
        clf0.fit(Xch_tr[Tch_tr==0], y_bin_ch[Tch_tr==0])

        mask_pos = ych_tr > 0
        if (mask_pos & (Tch_tr==1)).sum() > 10 and (mask_pos & (Tch_tr==0)).sum() > 10:
            reg1.fit(Xch_tr[mask_pos & (Tch_tr==1)], np.log1p(ych_tr[mask_pos & (Tch_tr==1)]))
            reg0.fit(Xch_tr[mask_pos & (Tch_tr==0)], np.log1p(ych_tr[mask_pos & (Tch_tr==0)]))
            e1 = np.expm1(reg1.predict(Xch_vl))
            e0 = np.expm1(reg0.predict(Xch_vl))
        else:
            e1 = np.zeros(mask_vl.sum()); e0 = np.zeros(mask_vl.sum())

        p1 = clf1.predict_proba(Xch_vl)[:,1]
        p0 = clf0.predict_proba(Xch_vl)[:,1]

        fold_pred[mask_vl] = p1*e1 - p0*e0

    oof_scores_perchan[val_idx] = fold_pred

elapsed_perchan = time.time() - t0
u10_perchan = register_result('Per-Channel Hurdle', y, oof_scores_perchan, T, elapsed_perchan)
np.save(f'{ARTIFACTS_DIR}/oof_perchannel.npy', oof_scores_perchan)

Training Model 6: Per-Channel Hurdle T-Learner...
  [Per-Channel Hurdle] uplift@10=0.0423 [0.0382, 0.0487]  Qini=0.0241  t=1008s


## 12. Model 7 — Optuna-tuned Hurdle T-Learner

Optuna напрямую оптимизирует OOF uplift@10. 40 trials, 3-fold (ускорение).

In [ ]:
# ─── MODEL 7: Optuna-tuned Hurdle T-Learner ──────────────────────────────────
print('Training Model 7: Optuna-tuned Hurdle T-Learner...')
t0 = time.time()

# Subsample для быстрого поиска (30% данных, стратифицировано)
from sklearn.model_selection import StratifiedShuffleSplit
sss = StratifiedShuffleSplit(n_splits=1, test_size=0.7, random_state=RANDOM_STATE)
opt_idx, _ = next(sss.split(train, make_stratify_col(train)))
train_opt = train.iloc[opt_idx].reset_index(drop=True)
y_opt = train_opt[TARGET_COL].values
T_opt = train_opt[TREATMENT_COL].values

# 2-fold на маленьком датасете для скорости
folds_opt = list(StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
                 .split(train_opt, make_stratify_col(train_opt)))

def hurdle_oof_fast(params, folds_list, df_sub, y_sub, T_sub, feat_cols):
    """OOF на произвольном датасете"""
    oof = np.zeros(len(df_sub))
    for _, (tr_idx, val_idx) in enumerate(folds_list):
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr = y_sub[tr_idx]; Ttr = T_sub[tr_idx]

        clf1 = lgb.LGBMClassifier(**params)
        clf0 = lgb.LGBMClassifier(**params)
        reg1 = lgb.LGBMRegressor(**params)
        reg0 = lgb.LGBMRegressor(**params)

        clf1.fit(X_tr[Ttr==1], (ytr[Ttr==1]>0).astype(int))
        clf0.fit(X_tr[Ttr==0], (ytr[Ttr==0]>0).astype(int))
        m1p = (Ttr==1)&(ytr>0); m0p = (Ttr==0)&(ytr>0)
        reg1.fit(X_tr[m1p], np.log1p(ytr[m1p]))
        reg0.fit(X_tr[m0p], np.log1p(ytr[m0p]))

        p1 = clf1.predict_proba(X_val)[:,1]
        p0 = clf0.predict_proba(X_val)[:,1]
        e1 = np.expm1(reg1.predict(X_val))
        e0 = np.expm1(reg0.predict(X_val))
        oof[val_idx] = p1*e1 - p0*e0

    y_bin_sub = (y_sub > 0).astype(int)
    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof

def objective(trial):
    params = dict(
        # Фиксируем n_estimators маленьким для скорости поиска
        n_estimators     = trial.suggest_int('n_est', 100, 400),
        learning_rate    = trial.suggest_float('lr', 0.02, 0.2, log=True),
        num_leaves       = trial.suggest_int('nl', 31, 127),
        min_child_samples= trial.suggest_int('mcs', 20, 100),
        subsample        = trial.suggest_float('ss', 0.6, 1.0),
        colsample_bytree = trial.suggest_float('cst', 0.6, 1.0),
        reg_alpha        = trial.suggest_float('ra', 1e-3, 2.0, log=True),
        reg_lambda       = trial.suggest_float('rl', 1e-3, 2.0, log=True),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    feat_set = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
    feat_cols = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP
    try:
        score, _ = hurdle_oof_fast(params, folds_opt, train_opt, y_opt, T_opt, feat_cols)
        return score
    except Exception:
        return -999.0

study = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE)
)
# timeout=600 = жёсткий лимит 10 минут, n_trials=50 — сколько успеет
study.optimize(objective, n_trials=50, timeout=600, show_progress_bar=True)

best = study.best_params
print(f'Best Optuna trial: uplift@10={study.best_value:.4f}  trials_done={len(study.trials)}')
print(f'Best params: {best}')

best_params = dict(
    # Масштабируем n_estimators для финального обучения на полных данных
    n_estimators     = min(best['n_est'] * 3, 1500),
    learning_rate    = best['lr'],
    num_leaves       = best['nl'],
    min_child_samples= best['mcs'],
    subsample        = best['ss'],
    colsample_bytree = best['cst'],
    reg_alpha        = best['ra'],
    reg_lambda       = best['rl'],
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
best_feat_cols = FEAT_COLS if best['feat_set']=='all' else FEAT_COLS_TOP

# Финальный OOF на полных данных с лучшими параметрами (5-fold)
print('Running final 5-fold OOF with best params on full train...')
_, oof_scores_opt = hurdle_oof_fast(best_params, folds, train, y, T, best_feat_cols)

elapsed_opt = time.time() - t0
u10_opt = register_result('Optuna Hurdle', y, oof_scores_opt, T, elapsed_opt)
np.save(f'{ARTIFACTS_DIR}/oof_optuna.npy', oof_scores_opt)
with open(f'{ARTIFACTS_DIR}/optuna_best_params.json','w') as f:
    json.dump({'params': best_params, 'feat_set': best['feat_set'],
               'val_uplift10': study.best_value}, f, indent=2)

Training Model 7: Optuna-tuned Hurdle T-Learner...


Best trial: 15. Best value: 17.2937:  44%|████▍     | 22/50 [10:27<13:18, 28.52s/it, 627.41/600 seconds]


Best Optuna trial: uplift@10=17.2937  trials_done=22
Best params: {'n_est': 266, 'lr': 0.02198724687450408, 'nl': 72, 'mcs': 35, 'ss': 0.643470122581265, 'cst': 0.7281904841416579, 'ra': 0.3217870390623861, 'rl': 0.1634894109910857, 'feat_set': 'top25_ext'}
Running final 5-fold OOF with best params on full train...
  [Optuna Hurdle] uplift@10=18.0536 [17.0641, 19.8211]  Qini=0.0325  t=895s


In [ ]:
# ─── Пересчёт RESULTS с правильной метрикой uplift@10 по rec_spend ───────────
RESULTS = []  # сброс

oof_files = {
    'T-Learner':        'oof_tlearner.npy',
    'Hurdle T-Learner': 'oof_hurdle.npy',
    'X-Learner':        'oof_xlearner.npy',
    'DR-Learner':       'oof_drlearner.npy',
    'Hurdle DR-Learner':'oof_hurdle_dr.npy',
    'Per-Channel Hurdle':'oof_perchannel.npy',
    'Optuna Hurdle':    'oof_optuna.npy',
    'X-Learner Optuna':    'oof_xlearner_opt.npy',
}

print(f"{'Model':<22} {'uplift@10':>10} {'CI80_lo':>10} {'CI80_hi':>10}")
print('-' * 55)

for model_name, fname in oof_files.items():
    fpath = f'{ARTIFACTS_DIR}/{fname}'
    if not os.path.exists(fpath):
        print(f'{model_name:<22}  — файл не найден, пропускаем')
        continue
    
    oof = np.load(fpath)
    u10, ci_lo, ci_hi = oof_uplift_at_k(y, oof, T, k=0.1, n_boot=100)
    
    RESULTS.append({
        'model':     model_name,
        'uplift@10': round(u10, 4),
        'ci80_lo':   round(ci_lo, 4),
        'ci80_hi':   round(ci_hi, 4),
    })
    print(f'{model_name:<22} {u10:>10.4f} {ci_lo:>10.4f} {ci_hi:>10.4f}')

# Сохраняем обновлённую таблицу
results_df = pd.DataFrame(RESULTS).sort_values('uplift@10', ascending=False)
results_df.to_csv(f'{ARTIFACTS_DIR}/model_comparison.csv', index=False)
print(f'\n✅ model_comparison.csv обновлён')

Model                   uplift@10    CI80_lo    CI80_hi
-------------------------------------------------------
T-Learner                 18.5035    16.8350    20.5097
Hurdle T-Learner          16.7450    15.6530    18.6956
X-Learner                 19.9622    18.2455    21.8872
DR-Learner                17.0512    15.3955    18.9810
Hurdle DR-Learner         10.3748     9.3049    11.5078
Per-Channel Hurdle        15.0752    13.7572    16.9268
Optuna Hurdle             18.0536    17.0641    19.8211
X-Learner Optuna          21.0739    19.4908    22.4540

✅ model_comparison.csv обновлён


In [ ]:
# ─── SUBMISSION: Обучение на всём train + скоринг теста ──────────────────────
print('Training final X-Learner on full train + scoring test...')
t0 = time.time()

from sklearn.linear_model import LogisticRegression

X_train_full = encode_X(train, best_feat_xl)
X_test_full  = encode_X(test,  best_feat_xl)

mask1_full = T == 1
mask0_full = T == 0

# Stage 1: outcome models на полном train
print('  Stage 1: fitting mu1, mu0...')
mu1_final = lgb.LGBMRegressor(**best_params1_xl)
mu0_final = lgb.LGBMRegressor(**best_params1_xl)
mu1_final.fit(X_train_full[mask1_full], y[mask1_full])
mu0_final.fit(X_train_full[mask0_full], y[mask0_full])

# Imputed CATE
D1_full = y[mask1_full] - mu0_final.predict(X_train_full[mask1_full])
D0_full = mu1_final.predict(X_train_full[mask0_full]) - y[mask0_full]

# Stage 2: CATE models
print('  Stage 2: fitting tau1, tau0...')
tau1_final = lgb.LGBMRegressor(**best_params2_xl)
tau0_final = lgb.LGBMRegressor(**best_params2_xl)
tau1_final.fit(X_train_full[mask1_full], D1_full)
tau0_final.fit(X_train_full[mask0_full], D0_full)

# Propensity на полном train
print(f'  Propensity: fitting {best_prop_xl}...')
if best_prop_xl == 'logreg':
    g_final = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
else:
    g_final = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                  n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
g_final.fit(X_train_full, T)

# Скоринг теста
g_test = g_final.predict_proba(X_test_full)[:, 1]
test_scores = (g_test * tau0_final.predict(X_test_full) +
               (1 - g_test) * tau1_final.predict(X_test_full))

# Формируем сабмит
submission = pd.DataFrame({
    'user_id': test['user_id'],
    'uplift':    test_scores
})

sub_path = f'{ARTIFACTS_DIR}/submission_xl_opt.csv'
submission.to_csv(sub_path, index=False)

elapsed_final = time.time() - t0
print(f'\n✅ Done in {elapsed_final:.0f}s')
print(f'✅ Submission saved: {sub_path}')
print(f'   Shape: {submission.shape}')
print(f'   uplift stats: min={test_scores.min():.4f}  '
      f'max={test_scores.max():.4f}  '
      f'mean={test_scores.mean():.4f}  '
      f'std={test_scores.std():.4f}')
print(f'\nTop-5 rows:')
print(submission.head())

Training final X-Learner on full train + scoring test...
  Stage 1: fitting mu1, mu0...
  Stage 2: fitting tau1, tau0...
  Propensity: fitting logreg...


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


KeyError: 'client_id'

In [ ]:
# ═══ Распределение adversarial scores (train_adv_probs) ═══════════════════════
import matplotlib.pyplot as plt
import numpy as np

train_adv_probs = oof_probs[y_adv == 0]  # только train-строки

fig, axes = plt.subplots(1, 3, figsize=(18, 5))

# ── 1. Гистограмма распределения скоров ──────────────────────────────────────
axes[0].hist(train_adv_probs, bins=100, color='#3498db', alpha=0.8, edgecolor='none')
axes[0].axvline(0.5,  color='red',    linestyle='--', linewidth=1.5, label='0.5')
axes[0].axvline(0.7,  color='orange', linestyle='--', linewidth=1.5, label='0.7')
axes[0].axvline(0.9,  color='green',  linestyle='--', linewidth=1.5, label='0.9')
for q in [0.8, 0.9, 0.95]:
    v = np.quantile(train_adv_probs, q)
    axes[0].axvline(v, color='gray', linestyle=':', linewidth=1, alpha=0.5)
axes[0].set_xlabel('P(is_test) — adversarial score')
axes[0].set_ylabel('Count')
axes[0].set_title('Distribution of Adversarial Scores\n(train строки)')
axes[0].legend(title='thresholds')

# ── 2. ECDF (накопительное распределение) ────────────────────────────────────
sorted_probs = np.sort(train_adv_probs)
ecdf = np.arange(1, len(sorted_probs) + 1) / len(sorted_probs)
axes[1].plot(sorted_probs, ecdf, color='#e74c3c', linewidth=2)
axes[1].axvline(0.5, color='red',    linestyle='--', linewidth=1.5, label='0.5')
axes[1].axvline(0.7, color='orange', linestyle='--', linewidth=1.5, label='0.7')
axes[1].axvline(0.9, color='green',  linestyle='--', linewidth=1.5, label='0.9')
axes[1].set_xlabel('Adversarial score')
axes[1].set_ylabel('ECDF')
axes[1].set_title('ECDF: сколько % строк ниже порога')
axes[1].legend()
axes[1].grid(alpha=0.3)

# ── 3. Доля строк по бакетам ──────────────────────────────────────────────────
thresholds = [0.3, 0.4, 0.5, 0.6, 0.7, 0.8, 0.9, 0.95, 0.99]
counts  = [(train_adv_probs >= t).sum() for t in thresholds]
pcts    = [c / len(train_adv_probs) * 100 for c in counts]
bars = axes[2].bar([str(t) for t in thresholds], pcts,
                   color=['#27ae60' if t >= 0.7 else '#3498db' for t in thresholds],
                   alpha=0.85)
for bar, c, p in zip(bars, counts, pcts):
    axes[2].text(bar.get_x() + bar.get_width()/2, bar.get_height() + 0.3,
                 f'{c:,}\n({p:.1f}%)', ha='center', va='bottom', fontsize=8)
axes[2].set_xlabel('Порог P(is_test) ≥')
axes[2].set_ylabel('% строк train')
axes[2].set_title('Сколько строк выше порога\n(кандидаты для sample_weight)')
axes[2].grid(axis='y', alpha=0.3)

plt.suptitle(f'Adversarial Score Analysis\n'
             f'train N={len(train_adv_probs):,}  '
             f'mean={train_adv_probs.mean():.3f}  '
             f'median={np.median(train_adv_probs):.3f}',
             fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/adversarial/adv_score_distribution.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Текстовая сводка ──────────────────────────────────────────────────────────
print(f'Всего train строк: {len(train_adv_probs):,}')
print(f'Mean score:   {train_adv_probs.mean():.4f}')
print(f'Median score: {np.median(train_adv_probs):.4f}')
print()
print('Распределение по порогам:')
for t, c, p in zip(thresholds, counts, pcts):
    bar_vis = '█' * int(p / 2)
    print(f'  score >= {t}: {c:>7,} строк  ({p:5.1f}%)  {bar_vis}')

print(f'\nРекомендуемый val-set (score >= 0.8): {(train_adv_probs >= 0.8).sum():,} строк '
      f'({(train_adv_probs >= 0.8).mean()*100:.1f}%)')

Всего train строк: 355,246
Mean score:   0.2364
Median score: 0.0971

Распределение по порогам:
  score >= 0.3: 127,008 строк  ( 35.8%)  █████████████████
  score >= 0.4: 107,171 строк  ( 30.2%)  ███████████████
  score >= 0.5:  85,300 строк  ( 24.0%)  ████████████
  score >= 0.6:  59,856 строк  ( 16.8%)  ████████
  score >= 0.7:  26,483 строк  (  7.5%)  ███
  score >= 0.8:     455 строк  (  0.1%)  
  score >= 0.9:       1 строк  (  0.0%)  
  score >= 0.95:       0 строк  (  0.0%)  
  score >= 0.99:       0 строк  (  0.0%)  

Рекомендуемый val-set (score >= 0.8): 455 строк (0.1%)


In [67]:
# ═══ Качество X-Learner на test-like vs train-like клиентах ══════════════════
# oof_preds = oof_scores_xl_opt  ← вот откуда берём
# train_adv_probs — из adversarial validation (y_adv == 0 часть)

oof_preds = oof_scores_xl_opt  # алиас для читаемости

THRESHOLD = 0.6

df_eval = pd.DataFrame({
    'oof_pred':  oof_preds,
    'treatment': train[TREATMENT_COL].values,
    'target':    train[TARGET_COL].values,
    'adv_score': train_adv_probs,
})

# ── Группы ───────────────────────────────────────────────────────────────────
groups = {
    'ALL train':              np.ones(len(df_eval), dtype=bool),
    f'test-like  (≥{THRESHOLD})': train_adv_probs >= THRESHOLD,
    f'train-like (<{THRESHOLD})': train_adv_probs <  THRESHOLD,
}
for q_lo, q_hi in [(0.0, 0.2), (0.2, 0.4), (0.4, 0.6), (0.6, 0.8), (0.8, 1.0)]:
    lo_v = np.quantile(train_adv_probs, q_lo)
    hi_v = np.quantile(train_adv_probs, q_hi)
    mask = (train_adv_probs >= lo_v) & (train_adv_probs < hi_v)
    groups[f'adv [{q_lo:.1f}-{q_hi:.1f}]'] = mask

results = {}
print(f'{"Группа":<32} {"N":>8} {"T%":>6} {"uplift@10":>12} {"uplift@20":>12}')
print('─' * 75)
for name, mask in groups.items():
    sub = df_eval[mask]
    if mask.sum() < 200:
        print(f'{name:<32} {"< 200 строк — пропуск":>45}')
        continue
    u10 = uplift_at_k_spend(
        sub['target'].values, sub['oof_pred'].values,
        sub['treatment'].values, k=0.10
    )
    u20 = uplift_at_k_spend(
        sub['target'].values, sub['oof_pred'].values,
        sub['treatment'].values, k=0.20
    )
    results[name] = {'n': mask.sum(), 'u10': u10, 'u20': u20,
                     't_pct': sub['treatment'].mean() * 100,
                     'adv_mean': sub['adv_score'].mean()}
    print(f'{name:<32} {mask.sum():>8,} {results[name]["t_pct"]:>5.1f}% {u10:>12.5f} {u20:>12.5f}')

# ── Визуализация ──────────────────────────────────────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

q_keys  = [k for k in results if k.startswith('adv')]
u10_vals = [results[k]['u10'] for k in q_keys]
u20_vals = [results[k]['u20'] for k in q_keys]
x = np.arange(len(q_keys))
baseline = results.get('ALL train', {}).get('u10', 0)

axes[0].bar(x - 0.2, u10_vals, width=0.4, label='uplift@10', color='#3498db', alpha=0.85)
axes[0].bar(x + 0.2, u20_vals, width=0.4, label='uplift@20', color='#e74c3c', alpha=0.85)
axes[0].axhline(baseline, color='gray', linestyle='--', linewidth=1.5,
                label=f'ALL baseline u@10={baseline:.4f}')
for xi, (u10, u20) in zip(x, zip(u10_vals, u20_vals)):
    axes[0].text(xi - 0.2, u10 + abs(baseline)*0.02, f'{u10:.4f}',
                 ha='center', va='bottom', fontsize=7, rotation=45)
axes[0].set_xticks(x); axes[0].set_xticklabels(q_keys, rotation=20, ha='right')
axes[0].set_title('Uplift@k по квантилям adversarial score\n(правый бакет = самые похожие на test)')
axes[0].set_ylabel('Uplift'); axes[0].legend(); axes[0].grid(axis='y', alpha=0.3)

# Scatter: adv_score vs oof_pred
sample_idx = np.random.choice(len(df_eval), size=min(15000, len(df_eval)), replace=False)
sc = axes[1].scatter(
    df_eval['adv_score'].iloc[sample_idx],
    df_eval['oof_pred'].iloc[sample_idx],
    c=df_eval['treatment'].iloc[sample_idx],
    cmap='coolwarm', alpha=0.15, s=4
)
axes[1].axvline(THRESHOLD, color='red', linestyle='--', linewidth=1.5,
                label=f'threshold={THRESHOLD}')
plt.colorbar(sc, ax=axes[1], label='treatment')
axes[1].set_xlabel('Adversarial score P(is_test)')
axes[1].set_ylabel('OOF uplift (X-Learner)')
axes[1].set_title('Adversarial score vs predicted CATE\n(раскраска: treatment=1 красный)')
axes[1].legend(); axes[1].grid(alpha=0.3)

plt.suptitle('X-Learner quality: test-like vs train-like clients', fontsize=13, y=1.02)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/adversarial/uplift_by_adv_score.png', dpi=150, bbox_inches='tight')
plt.show()

# ── Итоговый вывод ────────────────────────────────────────────────────────────
u_all  = results.get('ALL train', {}).get('u10', 0)
u_test = results.get(f'test-like  (≥{THRESHOLD})', {}).get('u10', 0)
u_tr   = results.get(f'train-like (<{THRESHOLD})', {}).get('u10', 0)
delta  = u_test - u_all

print(f'\n{"─"*50}')
print(f'uplift@10 ALL:        {u_all:.5f}')
print(f'uplift@10 test-like:  {u_test:.5f}  (Δ = {delta:+.5f})')
print(f'uplift@10 train-like: {u_tr:.5f}')
print()
if delta > 0.002:
    print('✅ Модель ЛУЧШЕ работает на test-like клиентах.')
    print('   → Фильтрация или sample_weight даст прирост на LB')
elif delta < -0.002:
    print('⚠️  Модель ХУЖЕ на test-like — shift мешает.')
    print('   → Попробуй rank-transform shift-фичей перед финальным обучением')
else:
    print('➡️  Разница незначительная (< 0.002).')
    print('   → sample_weight не поможет, лучше потратить время на другое')

Группа                                  N     T%    uplift@10    uplift@20
───────────────────────────────────────────────────────────────────────────
ALL train                         355,246  49.6%     21.07387     12.38159
test-like  (≥0.6)                  59,856  50.0%     18.47672     11.39333
train-like (<0.6)                 295,390  49.5%     21.57329     13.09991
adv [0.0-0.2]                      71,049  49.8%     17.04431      9.15907
adv [0.2-0.4]                      71,049  49.7%     10.21349      7.19651
adv [0.4-0.6]                      71,049  49.3%     30.80466     18.42745
adv [0.6-0.8]                      71,049  48.9%     31.76265     17.80261
adv [0.8-1.0]                      71,049  50.1%     18.08213     11.36648

──────────────────────────────────────────────────
uplift@10 ALL:        21.07387
uplift@10 test-like:  18.47672  (Δ = -2.59715)
uplift@10 train-like: 21.57329

⚠️  Модель ХУЖЕ на test-like — shift мешает.
   → Попробуй rank-transform shift-фичей п

In [66]:
# Быстрая проверка стабильности
sub_testlike = df_eval[train_adv_probs >= 0.8]
boot_scores = []
for _ in range(200):
    s = sub_testlike.sample(len(sub_testlike), replace=True)
    boot_scores.append(uplift_at_k_spend(
        s['target'].values, s['oof_pred'].values, s['treatment'].values, k=0.1))
print(f'Bootstrap CI: {np.percentile(boot_scores, 5):.1f} – {np.percentile(boot_scores, 95):.1f}')

Bootstrap CI: 11.7 – 166.7


In [42]:
# Формируем сабмит
submission = pd.DataFrame({
    'user_id': test['user_id'],
    'UPLIFT_SCORE':   test_scores
})

sub_path = f'{ARTIFACTS_DIR}/submission_xl_opt.csv'
submission.to_csv(sub_path, index=False)

elapsed_final = time.time() - t0
print(f'\n✅ Done in {elapsed_final:.0f}s')
print(f'✅ Submission saved: {sub_path}')
print(f'   Shape: {submission.shape}')
print(f'   uplift stats: min={test_scores.min():.4f}  '
      f'max={test_scores.max():.4f}  '
      f'mean={test_scores.mean():.4f}  '
      f'std={test_scores.std():.4f}')
print(f'\nTop-5 rows:')
print(submission.head())


✅ Done in 349s
✅ Submission saved: pilot_artifacts/submission_xl_opt.csv
   Shape: (118414, 2)
   uplift stats: min=-69.4020  max=379.7554  mean=3.2991  std=10.7458

Top-5 rows:
   user_id  UPLIFT_SCORE
0   356862      2.678280
1   449898      1.258673
2   449235      0.119192
3   432555     74.307838
4   460513      1.361311


In [69]:
# ═══ MODEL 10: Optuna X-Learner v2 — val на overlap-сегменте (adv ≥ 0.4) ════
print('Training Model 10: Optuna X-Learner v2 (overlap val)...')
t0 = time.time()

# ── 0. Overlap val-mask: adv_score ∈ [0.4, 1.0] ──────────────────────────────
ADV_VAL_THRESH = 0.4
overlap_mask = train_adv_probs >= ADV_VAL_THRESH
print(f'Overlap val segment (adv ≥ {ADV_VAL_THRESH}): '
      f'{overlap_mask.sum():,} строк ({overlap_mask.mean()*100:.1f}%)')

# ── 1. Subsample для поиска (40%) — стратифицируем по overlap+стратам ─────────
sss3 = StratifiedShuffleSplit(n_splits=1, test_size=0.6, random_state=RANDOM_STATE+2)
opt3_idx, _ = next(sss3.split(train, make_stratify_col(train)))
train_opt3   = train.iloc[opt3_idx].reset_index(drop=True)
y_opt3       = train_opt3[TARGET_COL].values
T_opt3       = train_opt3[TREATMENT_COL].values
adv_opt3     = train_adv_probs[opt3_idx]  # adv scores для subsample

# Val-маска внутри subsample
overlap_mask3 = adv_opt3 >= ADV_VAL_THRESH
print(f'  Subsample overlap: {overlap_mask3.sum():,} строк')

folds_opt3 = list(
    StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
    .split(train_opt3, make_stratify_col(train_opt3))
)

# ── 2. OOF с оценкой только на overlap-сегменте ───────────────────────────────
def xlearner_oof_v2(params1, params2, prop_model, objective1, objective2,
                    folds_list, df_sub, y_sub, T_sub, feat_cols,
                    adv_mask_sub, sample_weight=None):
    """
    X-Learner OOF v2:
      - objective1/2 — варьируемые loss-функции
      - оценка uplift@10 только на overlap-сегменте (adv_mask_sub)
      - опционально sample_weight на Stage1
    """
    oof = np.zeros(len(df_sub))

    for tr_idx, val_idx in folds_list:
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]
        sw_tr = sample_weight[tr_idx] if sample_weight is not None else None

        mask1 = Ttr == 1
        mask0 = Ttr == 0

        # Stage 1 — outcome models
        mu1 = lgb.LGBMRegressor(**params1, objective=objective1)
        mu0 = lgb.LGBMRegressor(**params1, objective=objective1)
        sw1 = sw_tr[mask1] if sw_tr is not None else None
        sw0 = sw_tr[mask0] if sw_tr is not None else None
        mu1.fit(X_tr[mask1], ytr[mask1], sample_weight=sw1)
        mu0.fit(X_tr[mask0], ytr[mask0], sample_weight=sw0)

        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]

        # Stage 2 — CATE models
        tau1 = lgb.LGBMRegressor(**params2, objective=objective2)
        tau0 = lgb.LGBMRegressor(**params2, objective=objective2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        # Propensity
        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        g.fit(X_tr, Ttr)
        g_val = g.predict_proba(X_val)[:, 1]

        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    # Оцениваем только на overlap val-строках
    val_in_overlap = adv_mask_sub  # маска по всему subsample
    score = uplift_at_k_spend(
        y_sub[val_in_overlap], oof[val_in_overlap],
        T_sub[val_in_overlap], k=0.1
    )
    return score, oof


def objective_xl_v2(trial):
    # ── Stage 1: outcome models (более сложные) ──────────────────────────────
    objective1 = trial.suggest_categorical('objective1', [
        'regression',           # MAE-like MSE
        'regression_l1',        # MAE — робастнее к выбросам (90% нулей!)
        'huber',                # Huber — компромисс MSE/MAE
        'quantile',             # квантильная регрессия
        'mape',                 # относительная ошибка
        'tweedie',              # для right-skewed с нулями
    ])
    params1 = dict(
        n_estimators      = trial.suggest_int('s1_n_est', 100, 600),
        learning_rate     = trial.suggest_float('s1_lr', 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int('s1_nl', 31, 300),
        max_depth         = trial.suggest_int('s1_md', 4, 10),
        min_child_samples = trial.suggest_int('s1_mcs', 10, 100),
        min_child_weight  = trial.suggest_float('s1_mcw', 1e-3, 10.0, log=True),
        subsample         = trial.suggest_float('s1_ss', 0.5, 1.0),
        subsample_freq    = trial.suggest_int('s1_ssf', 0, 5),
        colsample_bytree  = trial.suggest_float('s1_cst', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 5.0, log=True),
        reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 5.0, log=True),
        extra_trees       = trial.suggest_categorical('s1_et', [True, False]),
        path_smooth       = trial.suggest_float('s1_ps', 0.0, 5.0),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    # tweedie нужен специальный параметр
    if objective1 == 'tweedie':
        params1['tweedie_variance_power'] = trial.suggest_float('s1_tvp', 1.0, 1.9)
    if objective1 == 'quantile':
        params1['alpha'] = trial.suggest_float('s1_alpha', 0.4, 0.6)
    if objective1 == 'huber':
        params1['alpha'] = trial.suggest_float('s1_huber_alpha', 0.7, 0.99)

    # ── Stage 2: CATE models (проще) ─────────────────────────────────────────
    objective2 = trial.suggest_categorical('objective2', [
        'regression',
        'regression_l1',
        'huber',
    ])
    params2 = dict(
        n_estimators      = trial.suggest_int('s2_n_est', 50, 400),
        learning_rate     = trial.suggest_float('s2_lr', 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int('s2_nl', 15, 150),
        max_depth         = trial.suggest_int('s2_md', 3, 8),
        min_child_samples = trial.suggest_int('s2_mcs', 10, 100),
        min_child_weight  = trial.suggest_float('s2_mcw', 1e-3, 10.0, log=True),
        subsample         = trial.suggest_float('s2_ss', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('s2_cst', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 5.0, log=True),
        reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 5.0, log=True),
        path_smooth       = trial.suggest_float('s2_ps', 0.0, 5.0),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    if objective2 == 'huber':
        params2['alpha'] = trial.suggest_float('s2_huber_alpha', 0.7, 0.99)

    # ── Архитектурные выборы ──────────────────────────────────────────────────
    prop_model  = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
    feat_set    = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
    feat_cols   = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP

    # sample_weight: None / overlap / adv_raw
    sw_mode = trial.suggest_categorical('sw_mode', ['none', 'overlap', 'adv_raw'])
    if sw_mode == 'none':
        sw = None
    elif sw_mode == 'overlap':
        p = adv_opt3.clip(0.05, 0.95)
        sw = (p * (1 - p) * 4)
        sw = sw / sw.mean()
    else:  # adv_raw
        sw = adv_opt3.clip(0.01, 0.99)
        sw = sw / sw.mean()

    try:
        score, _ = xlearner_oof_v2(
            params1, params2, prop_model, objective1, objective2,
            folds_opt3, train_opt3, y_opt3, T_opt3, feat_cols,
            overlap_mask3, sample_weight=sw
        )
        return score
    except Exception as e:
        print(f'Trial failed: {e}')
        return -999.0


# ── 3. Optuna search ──────────────────────────────────────────────────────────
study_xl2 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=15),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=15, n_warmup_steps=0)
)
study_xl2.optimize(objective_xl_v2, n_trials=50, timeout=3600, show_progress_bar=True)

best_xl2 = study_xl2.best_params
print(f'\nBest v2 trial: uplift@10(overlap)={study_xl2.best_value:.4f}  '
      f'trials={len(study_xl2.trials)}')
print(f'objective1={best_xl2["objective1"]},  objective2={best_xl2["objective2"]}')
print(f'prop_model={best_xl2["prop_model"]},  feat_set={best_xl2["feat_set"]}')
print(f'sw_mode={best_xl2["sw_mode"]}')

# ── 4. Финальный OOF на полных данных (5-fold) ────────────────────────────────
def scale_params_v2(p, prefix, scale=2.5):
    return dict(
        n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 2000),
        learning_rate     = p[f'{prefix}_lr'],
        num_leaves        = p[f'{prefix}_nl'],
        max_depth         = p[f'{prefix}_md'],
        min_child_samples = p[f'{prefix}_mcs'],
        min_child_weight  = p[f'{prefix}_mcw'],
        subsample         = p[f'{prefix}_ss'],
        colsample_bytree  = p[f'{prefix}_cst'],
        reg_alpha         = p[f'{prefix}_ra'],
        reg_lambda        = p[f'{prefix}_rl'],
        path_smooth       = p[f'{prefix}_ps'],
        extra_trees       = p.get(f'{prefix}_et', False),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )

best_p1_xl2  = scale_params_v2(best_xl2, 's1')
best_p2_xl2  = scale_params_v2(best_xl2, 's2')
best_obj1    = best_xl2['objective1']
best_obj2    = best_xl2['objective2']
best_prop2   = best_xl2['prop_model']
best_feat2   = FEAT_COLS if best_xl2['feat_set'] == 'all' else FEAT_COLS_TOP
best_sw_mode = best_xl2['sw_mode']

# Строим sample_weight для полного трейна
if best_sw_mode == 'none':
    final_sw = None
elif best_sw_mode == 'overlap':
    p = train_adv_probs.clip(0.05, 0.95)
    final_sw = (p * (1 - p) * 4)
    final_sw = final_sw / final_sw.mean()
else:
    final_sw = train_adv_probs.clip(0.01, 0.99)
    final_sw = final_sw / final_sw.mean()

# tweedie/quantile/huber доп-параметры
for extra_key in ['s1_tvp', 's1_alpha', 's1_huber_alpha']:
    if extra_key in best_xl2:
        param_name = {'s1_tvp': 'tweedie_variance_power',
                      's1_alpha': 'alpha',
                      's1_huber_alpha': 'alpha'}[extra_key]
        best_p1_xl2[param_name] = best_xl2[extra_key]
for extra_key in ['s2_huber_alpha']:
    if extra_key in best_xl2:
        best_p2_xl2['alpha'] = best_xl2[extra_key]

print('\nRunning final 5-fold OOF on full train...')
_, oof_scores_xl2 = xlearner_oof_v2(
    best_p1_xl2, best_p2_xl2, best_prop2, best_obj1, best_obj2,
    folds, train, y, T, best_feat2,
    overlap_mask, sample_weight=final_sw
)

elapsed_xl2 = time.time() - t0
u10_xl2 = uplift_at_k_spend(y, oof_scores_xl2, T, k=0.1)
u20_xl2 = uplift_at_k_spend(y, oof_scores_xl2, T, k=0.2)
print(f'Optuna XL v2 (overlap-val):  uplift@10={u10_xl2:.4f}  uplift@20={u20_xl2:.4f}  ({elapsed_xl2:.0f}s)')

np.save(f'{ARTIFACTS_DIR}/oof_xlearner_opt_v2.npy', oof_scores_xl2)

# ── 5. Предсказания на тесте + сабмит ────────────────────────────────────────
print('\nPredicting on test...')
X_train_full = encode_X(train, best_feat2)
X_test_full  = encode_X(test,  best_feat2)

# Финальные модели обучаем на всём трейне
mu1_f = lgb.LGBMRegressor(**best_p1_xl2, objective=best_obj1)
mu0_f = lgb.LGBMRegressor(**best_p1_xl2, objective=best_obj1)
mask1_f = T == 1; mask0_f = T == 0
sw1_f = final_sw[mask1_f] if final_sw is not None else None
sw0_f = final_sw[mask0_f] if final_sw is not None else None
mu1_f.fit(X_train_full[mask1_f], y[mask1_f], sample_weight=sw1_f)
mu0_f.fit(X_train_full[mask0_f], y[mask0_f], sample_weight=sw0_f)

D1_f = y[mask1_f] - mu0_f.predict(X_train_full[mask1_f])
D0_f = mu1_f.predict(X_train_full[mask0_f]) - y[mask0_f]

tau1_f = lgb.LGBMRegressor(**best_p2_xl2, objective=best_obj2)
tau0_f = lgb.LGBMRegressor(**best_p2_xl2, objective=best_obj2)
tau1_f.fit(X_train_full[mask1_f], D1_f)
tau0_f.fit(X_train_full[mask0_f], D0_f)

if best_prop2 == 'logreg':
    g_f = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
else:
    g_f = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                              n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
g_f.fit(X_train_full, T)
g_test = g_f.predict_proba(X_test_full)[:, 1]

test_scores = g_test * tau0_f.predict(X_test_full) + (1 - g_test) * tau1_f.predict(X_test_full)

# Сабмит
submission = pd.DataFrame({
    'user_id':     test['user_id'],
    'UPLIFT_SCORE': test_scores
})
sub_path = f'{ARTIFACTS_DIR}/submission_xl_v2.csv'
submission.to_csv(sub_path, index=False)

elapsed_final = time.time() - t0
print(f'\n✅ Done in {elapsed_final:.0f}s')
print(f'✅ Submission saved: {sub_path}')
print(f'   Shape: {submission.shape}')
print(f'   uplift stats: min={test_scores.min():.4f}  max={test_scores.max():.4f}  '
      f'mean={test_scores.mean():.4f}  std={test_scores.std():.4f}')
print(f'\nTop-5 rows:')
print(submission.head())

# Сохраняем параметры
with open(f'{ARTIFACTS_DIR}/optuna_xl_v2_params.json', 'w') as f:
    json.dump({
        'params1': best_p1_xl2, 'params2': best_p2_xl2,
        'objective1': best_obj1, 'objective2': best_obj2,
        'prop_model': best_prop2, 'feat_set': best_xl2['feat_set'],
        'sw_mode': best_sw_mode,
        'val_segment': f'adv >= {ADV_VAL_THRESH}',
        'search_uplift10_overlap': study_xl2.best_value,
        'oof_uplift10_full': float(u10_xl2),
        'trials_completed': len(study_xl2.trials)
    }, f, indent=2)

Training Model 10: Optuna X-Learner v2 (overlap val)...
Overlap val segment (adv ≥ 0.4): 107,171 строк (30.2%)
  Subsample overlap: 42,783 строк


Best trial: 0. Best value: 19.2774:   2%|▏         | 1/50 [00:20<17:05, 20.93s/it, 20.72/3600 seconds]

[W 2026-05-22 15:21:13,705] Trial 1 failed with parameters: {'objective1': 'huber', 's1_n_est': 360, 's1_lr': 0.05143828405076928, 's1_nl': 80, 's1_md': 10, 's1_mcs': 80, 's1_mcw': 5.727904470799624, 's1_ss': 0.9474136752138245, 's1_ssf': 3, 's1_cst': 0.9531245410138701, 's1_ra': 0.002124863863243128, 's1_rl': 0.00530804663077594, 's1_et': False, 's1_ps': 1.9433864484474102, 's1_huber_alpha': 0.7786912192144297, 'objective2': 'regression', 's2_n_est': 240, 's2_lr': 0.015252697030515175, 's2_nl': 124, 's2_md': 3, 's2_mcs': 99, 's2_mcw': 1.227380098785297, 's2_ss': 0.5993578407670862, 's2_cst': 0.4033132702741615, 's2_ra': 1.038406417693451, 's2_rl': 0.4117599592262879, 's2_ps': 3.6450358402049368, 'prop_model': 'logreg', 'feat_set': 'all', 'sw_mode': 'none'} because of the following error: KeyboardInterrupt().
Traceback (most recent call last):
  File "/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/optuna/study/_optimize.py", line 206, in _run_trial
    

KeyboardInterrupt: 

In [72]:
# ═══ MODEL 11: Optuna X-Learner v3 — weighted bootstrap CI lower bound ════
print('Training Model 11: Optuna X-Learner v3 (weighted bootstrap CI)...')
t0 = time.time()

ADV_VAL_THRESH = 0.3          # строки ниже порога почти не попадут в бутстрап
N_BOOTSTRAP    = 200          # итераций бутстрапа — баланс скорость/стабильность
CI_ALPHA       = 0.80         # 80% CI как у организаторов
RANDOM_STATE_BS = RANDOM_STATE + 99

overlap_mask = train_adv_probs >= ADV_VAL_THRESH
print(f'Overlap mask (adv ≥ {ADV_VAL_THRESH}): '
      f'{overlap_mask.sum():,} строк ({overlap_mask.mean()*100:.1f}%)')

# ── Subsample для Optuna (40% трейна) ────────────────────────────────────────
sss4 = StratifiedShuffleSplit(n_splits=1, test_size=0.6, random_state=RANDOM_STATE+3)
opt4_idx, _ = next(sss4.split(train, make_stratify_col(train)))
train_opt4   = train.iloc[opt4_idx].reset_index(drop=True)
y_opt4       = train_opt4[TARGET_COL].values
T_opt4       = train_opt4[TREATMENT_COL].values
adv_opt4     = train_adv_probs[opt4_idx]

folds_opt4 = list(
    StratifiedKFold(n_splits=3, shuffle=True, random_state=RANDOM_STATE)
    .split(train_opt4, make_stratify_col(train_opt4))
)


# ── Weighted bootstrap uplift@k ───────────────────────────────────────────────
def uplift_at_k_weighted_bootstrap(y, scores, treatment, adv_weights,
                                   k=0.1, n_boot=N_BOOTSTRAP,
                                   ci_alpha=CI_ALPHA, seed=RANDOM_STATE_BS):
    """
    Бутстрап uplift@k с весами по adv_score.
    Возвращает (point_estimate, lower_ci, upper_ci).
    
    Веса adv_weights — вероятность попасть в бутстрап-выборку:
    строки похожие на тест (высокий adv_score) семплируются чаще.
    """
    rng = np.random.default_rng(seed)
    n   = len(y)
    
    # Нормализуем веса → вероятности
    w = adv_weights.clip(0.01, 1.0)
    w = w / w.sum()
    
    boot_scores = np.empty(n_boot)
    for i in range(n_boot):
        idx = rng.choice(n, size=n, replace=True, p=w)
        boot_scores[i] = uplift_at_k_spend(
            y[idx], scores[idx], treatment[idx], k=k
        )
    
    lower = np.percentile(boot_scores, (1 - ci_alpha) / 2 * 100)
    upper = np.percentile(boot_scores, (1 + ci_alpha) / 2 * 100)
    point = uplift_at_k_spend(y, scores, treatment, k=k)
    return point, lower, upper


# ── OOF + метрика ─────────────────────────────────────────────────────────────
def xlearner_oof_v3(params1, params2, prop_model, objective1, objective2,
                    folds_list, df_sub, y_sub, T_sub, feat_cols,
                    adv_w, sample_weight=None):
    """
    X-Learner OOF v3:
      - полный OOF без маски
      - метрика = нижняя граница weighted bootstrap CI
      - веса по adv_score при бутстрапе
    """
    oof = np.zeros(len(df_sub))

    for tr_idx, val_idx in folds_list:
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]
        sw_tr = sample_weight[tr_idx] if sample_weight is not None else None

        mask1 = Ttr == 1
        mask0 = Ttr == 0

        mu1 = lgb.LGBMRegressor(**params1, objective=objective1)
        mu0 = lgb.LGBMRegressor(**params1, objective=objective1)
        mu1.fit(X_tr[mask1], ytr[mask1],
                sample_weight=sw_tr[mask1] if sw_tr is not None else None)
        mu0.fit(X_tr[mask0], ytr[mask0],
                sample_weight=sw_tr[mask0] if sw_tr is not None else None)

        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]

        tau1 = lgb.LGBMRegressor(**params2, objective=objective2)
        tau0 = lgb.LGBMRegressor(**params2, objective=objective2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        g.fit(X_tr, Ttr)
        g_val = g.predict_proba(X_val)[:, 1]

        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    # Метрика: нижняя граница weighted bootstrap CI
    point, lower, upper = uplift_at_k_weighted_bootstrap(
        y_sub, oof, T_sub, adv_w
    )
    return lower, point, upper, oof


# ── Optuna objective ──────────────────────────────────────────────────────────
def objective_xl_v3(trial):
    objective1 = trial.suggest_categorical('objective1', [
        'regression', 'regression_l1', 'huber', 'quantile', 'tweedie',
    ])
    params1 = dict(
        n_estimators      = trial.suggest_int('s1_n_est', 100, 600),
        learning_rate     = trial.suggest_float('s1_lr', 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int('s1_nl', 31, 256),
        max_depth         = trial.suggest_int('s1_md', 4, 10),
        min_child_samples = trial.suggest_int('s1_mcs', 10, 100),
        min_child_weight  = trial.suggest_float('s1_mcw', 1e-3, 10.0, log=True),
        subsample         = trial.suggest_float('s1_ss', 0.5, 1.0),
        subsample_freq    = trial.suggest_int('s1_ssf', 0, 5),
        colsample_bytree  = trial.suggest_float('s1_cst', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 5.0, log=True),
        reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 5.0, log=True),
        extra_trees       = trial.suggest_categorical('s1_et', [True, False]),
        path_smooth       = trial.suggest_float('s1_ps', 0.0, 5.0),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    if objective1 == 'tweedie':
        params1['tweedie_variance_power'] = trial.suggest_float('s1_tvp', 1.0, 1.9)
    if objective1 == 'quantile':
        params1['alpha'] = trial.suggest_float('s1_alpha', 0.4, 0.6)
    if objective1 == 'huber':
        params1['alpha'] = trial.suggest_float('s1_huber_alpha', 0.7, 0.99)

    objective2 = trial.suggest_categorical('objective2', [
        'regression', 'regression_l1', 'huber',
    ])
    params2 = dict(
        n_estimators      = trial.suggest_int('s2_n_est', 50, 400),
        learning_rate     = trial.suggest_float('s2_lr', 0.01, 0.2, log=True),
        num_leaves        = trial.suggest_int('s2_nl', 15, 150),
        max_depth         = trial.suggest_int('s2_md', 3, 8),
        min_child_samples = trial.suggest_int('s2_mcs', 10, 100),
        min_child_weight  = trial.suggest_float('s2_mcw', 1e-3, 10.0, log=True),
        subsample         = trial.suggest_float('s2_ss', 0.5, 1.0),
        colsample_bytree  = trial.suggest_float('s2_cst', 0.4, 1.0),
        reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 5.0, log=True),
        reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 5.0, log=True),
        path_smooth       = trial.suggest_float('s2_ps', 0.0, 5.0),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    if objective2 == 'huber':
        params2['alpha'] = trial.suggest_float('s2_huber_alpha', 0.7, 0.99)

    prop_model = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
    feat_set   = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
    feat_cols  = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP

    sw_mode = trial.suggest_categorical('sw_mode', ['none', 'overlap', 'adv_raw'])
    if sw_mode == 'none':
        sw = None
    elif sw_mode == 'overlap':
        p  = adv_opt4.clip(0.05, 0.95)
        sw = (p * (1 - p) * 4) / ((adv_opt4.clip(0.05, 0.95) * (1 - adv_opt4.clip(0.05, 0.95)) * 4).mean())
    else:
        sw = adv_opt4.clip(0.01, 0.99)
        sw = sw / sw.mean()

    try:
        lower, point, upper, _ = xlearner_oof_v3(
            params1, params2, prop_model, objective1, objective2,
            folds_opt4, train_opt4, y_opt4, T_opt4, feat_cols,
            adv_opt4, sample_weight=sw
        )
        # Логируем point и upper как user_attrs для анализа
        trial.set_user_attr('uplift_point', point)
        trial.set_user_attr('uplift_upper', upper)
        return lower  # ← оптимизируем нижнюю границу CI
    except Exception as e:
        print(f'Trial failed: {e}')
        return -999.0


# ── Optuna search ─────────────────────────────────────────────────────────────
study_xl3 = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE, n_startup_trials=20),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=20, n_warmup_steps=0)
)
study_xl3.optimize(objective_xl_v3, n_trials=60, timeout=3600, show_progress_bar=True)

best_xl3 = study_xl3.best_params
best_trial3 = study_xl3.best_trial
print(f'\nBest v3 trial:')
print(f'  CI lower (optimized) = {study_xl3.best_value:.4f}')
print(f'  point estimate       = {best_trial3.user_attrs["uplift_point"]:.4f}')
print(f'  CI upper             = {best_trial3.user_attrs["uplift_upper"]:.4f}')
print(f'  objective1={best_xl3["objective1"]}, objective2={best_xl3["objective2"]}')
print(f'  prop_model={best_xl3["prop_model"]}, feat_set={best_xl3["feat_set"]}')
print(f'  sw_mode={best_xl3["sw_mode"]}')


# ── Финальный OOF на полных данных (5-fold) ───────────────────────────────────
def scale_params_v3(p, prefix, scale=2.5):
    d = dict(
        n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 2000),
        learning_rate     = p[f'{prefix}_lr'],
        num_leaves        = p[f'{prefix}_nl'],
        max_depth         = p[f'{prefix}_md'],
        min_child_samples = p[f'{prefix}_mcs'],
        min_child_weight  = p[f'{prefix}_mcw'],
        subsample         = p[f'{prefix}_ss'],
        colsample_bytree  = p[f'{prefix}_cst'],
        reg_alpha         = p[f'{prefix}_ra'],
        reg_lambda        = p[f'{prefix}_rl'],
        path_smooth       = p[f'{prefix}_ps'],
        extra_trees       = p.get(f'{prefix}_et', False),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    if f'{prefix}_ssf' in p:
        d['subsample_freq'] = p[f'{prefix}_ssf']
    return d

best_p1_xl3 = scale_params_v3(best_xl3, 's1')
best_p2_xl3 = scale_params_v3(best_xl3, 's2')
best_obj1   = best_xl3['objective1']
best_obj2   = best_xl3['objective2']
best_prop3  = best_xl3['prop_model']
best_feat3  = FEAT_COLS if best_xl3['feat_set'] == 'all' else FEAT_COLS_TOP
best_sw3    = best_xl3['sw_mode']

for extra_key, param_name in [('s1_tvp', 'tweedie_variance_power'),
                               ('s1_alpha', 'alpha'),
                               ('s1_huber_alpha', 'alpha')]:
    if extra_key in best_xl3:
        best_p1_xl3[param_name] = best_xl3[extra_key]
if 's2_huber_alpha' in best_xl3:
    best_p2_xl3['alpha'] = best_xl3['s2_huber_alpha']

if best_sw3 == 'none':
    final_sw3 = None
elif best_sw3 == 'overlap':
    p = train_adv_probs.clip(0.05, 0.95)
    final_sw3 = (p * (1 - p) * 4)
    final_sw3 = final_sw3 / final_sw3.mean()
else:
    final_sw3 = train_adv_probs.clip(0.01, 0.99)
    final_sw3 = final_sw3 / final_sw3.mean()

print('\nRunning final 5-fold OOF on full train...')
lower_f, point_f, upper_f, oof_scores_xl3 = xlearner_oof_v3(
    best_p1_xl3, best_p2_xl3, best_prop3, best_obj1, best_obj2,
    folds, train, y, T, best_feat3,
    train_adv_probs, sample_weight=final_sw3
)
print(f'Final OOF  uplift@10: point={point_f:.4f}  '
      f'CI=[{lower_f:.4f}, {upper_f:.4f}]  (80% weighted bootstrap)')

u20_xl3 = uplift_at_k_spend(y, oof_scores_xl3, T, k=0.2)
print(f'Final OOF  uplift@20: {u20_xl3:.4f}')

np.save(f'{ARTIFACTS_DIR}/oof_xlearner_opt_v3.npy', oof_scores_xl3)


# ── Предсказания на тесте ─────────────────────────────────────────────────────
print('\nPredicting on test...')
X_train_full = encode_X(train, best_feat3)
X_test_full  = encode_X(test,  best_feat3)

mu1_f = lgb.LGBMRegressor(**best_p1_xl3, objective=best_obj1)
mu0_f = lgb.LGBMRegressor(**best_p1_xl3, objective=best_obj1)
mask1_f = T == 1; mask0_f = T == 0
mu1_f.fit(X_train_full[mask1_f], y[mask1_f],
          sample_weight=final_sw3[mask1_f] if final_sw3 is not None else None)
mu0_f.fit(X_train_full[mask0_f], y[mask0_f],
          sample_weight=final_sw3[mask0_f] if final_sw3 is not None else None)

D1_f = y[mask1_f] - mu0_f.predict(X_train_full[mask1_f])
D0_f = mu1_f.predict(X_train_full[mask0_f]) - y[mask0_f]

tau1_f = lgb.LGBMRegressor(**best_p2_xl3, objective=best_obj2)
tau0_f = lgb.LGBMRegressor(**best_p2_xl3, objective=best_obj2)
tau1_f.fit(X_train_full[mask1_f], D1_f)
tau0_f.fit(X_train_full[mask0_f], D0_f)

if best_prop3 == 'logreg':
    g_f = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
else:
    g_f = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                              n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
g_f.fit(X_train_full, T)
g_test = g_f.predict_proba(X_test_full)[:, 1]

test_scores = g_test * tau0_f.predict(X_test_full) + (1 - g_test) * tau1_f.predict(X_test_full)

submission = pd.DataFrame({
    'user_id':      test['user_id'],
    'UPLIFT_SCORE': test_scores
})
sub_path = f'{ARTIFACTS_DIR}/submission_xl_v3.csv'
submission.to_csv(sub_path, index=False)

elapsed = time.time() - t0
print(f'\n✅ Done in {elapsed:.0f}s  |  saved: {sub_path}')
print(f'   score stats: min={test_scores.min():.3f}  max={test_scores.max():.3f}  '
      f'mean={test_scores.mean():.3f}  std={test_scores.std():.3f}')

with open(f'{ARTIFACTS_DIR}/optuna_xl_v3_params.json', 'w') as f:
    json.dump({
        'params1': best_p1_xl3, 'params2': best_p2_xl3,
        'objective1': best_obj1, 'objective2': best_obj2,
        'prop_model': best_prop3, 'feat_set': best_xl3['feat_set'],
        'sw_mode': best_sw3,
        'bootstrap_n': N_BOOTSTRAP, 'ci_alpha': CI_ALPHA,
        'adv_val_thresh': ADV_VAL_THRESH,
        'search_ci_lower': study_xl3.best_value,
        'final_oof_point': float(point_f),
        'final_oof_ci_lower': float(lower_f),
        'final_oof_ci_upper': float(upper_f),
        'trials_completed': len(study_xl3.trials)
    }, f, indent=2)

Training Model 11: Optuna X-Learner v3 (weighted bootstrap CI)...
Overlap mask (adv ≥ 0.3): 127,008 строк (35.8%)


Best trial: 0. Best value: 9.91028:   5%|▌         | 3/60 [02:14<39:25, 41.51s/it, 134.53/3600 seconds]/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/prep


Best v3 trial:
  CI lower (optimized) = 19.5076
  point estimate       = 20.2161
  CI upper             = 25.1382
  objective1=tweedie, objective2=regression_l1
  prop_model=logreg, feat_set=top25_ext
  sw_mode=overlap

Running final 5-fold OOF on full train...
Final OOF  uplift@10: point=19.8668  CI=[19.6425, 22.8986]  (80% weighted bootstrap)
Final OOF  uplift@20: 10.9139

Predicting on test...


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(



✅ Done in 4018s  |  saved: pilot_artifacts/submission_xl_v3.csv
   score stats: min=-49.147  max=273.136  mean=1.619  std=7.140


In [43]:
submission.shape

(118414, 2)

In [39]:
test

,user_id,communication_type,age,n_days_life,gendercalc,n_plastic_card,n_virtual_card,rto,atv,mtv,...,cat7_share,cat6_share,cat5_share,mkt_resp_rate,mkt_view_rate,recency_ratio,spend_cv,trn_density,cat_breadth_ratio,cat7_vs_last_visit
0,356862,0,87.0,1859.0,1.0,2.0,1.0,14963.0,416.0,374.0,...,0.038094,0.038094,0.074183,0.000000,0.000000,3.921567,0.629808,7.068316,2.952381,2.000000
1,449898,0,49.0,327.0,0.0,NaN,1.0,7715.0,701.0,546.0,...,NaN,NaN,0.012962,0.043478,0.043478,NaN,0.656205,12.278287,2.500000,NaN
2,449235,1,25.0,1878.0,0.0,NaN,1.0,19655.0,1638.0,1678.0,...,NaN,NaN,0.017298,0.036364,0.218182,NaN,0.619658,2.332268,3.100000,NaN
3,432555,1,56.0,510.0,0.0,NaN,1.0,24850.0,777.0,523.0,...,0.059759,0.069819,0.097183,0.108108,0.432432,7.725319,1.268983,22.901961,1.842105,4.499999
4,460513,2,42.0,386.0,0.0,NaN,1.0,1802.0,450.0,462.0,...,NaN,NaN,NaN,0.000000,0.384615,6.655755,0.328889,3.782383,1.500000,30.499992
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
118409,403941,2,45.0,1245.0,0.0,NaN,1.0,6963.0,870.0,997.0,...,0.017234,0.017234,0.017234,0.136364,0.431818,4.728950,0.544828,2.345382,1.785714,10.249997
118410,380729,0,26.0,251.0,1.0,NaN,1.0,13367.0,371.0,309.0,...,NaN,0.009725,0.035760,0.028986,0.463768,111.274455,0.749326,52.350597,3.217391,56.749986
118411,365578,2,57.0,500.0,1.0,NaN,1.0,3694.0,528.0,561.0,...,NaN,NaN,0.104223,0.050000,0.300000,NaN,0.511364,5.110000,1.933333,NaN
118412,406179,2,33.0,2066.0,0.0,1.0,1.0,9711.0,809.0,673.0,...,NaN,NaN,0.032952,0.071429,0.809524,17.090906,0.454883,2.120039,2.222222,23.499994


In [71]:
# ═══════════════════════════════════════════════════════════════════════════════
# ADVERSARIAL VALIDATION — полная ячейка (gain everywhere)
# ═══════════════════════════════════════════════════════════════════════════════
import warnings, os
import numpy as np
import pandas as pd
import lightgbm as lgb
import matplotlib.pyplot as plt
from scipy.stats import ks_2samp, gaussian_kde
from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import roc_curve, auc, roc_auc_score

warnings.filterwarnings('ignore')
os.makedirs(f'{ARTIFACTS_DIR}/adversarial', exist_ok=True)

FEAT_COLS_ADV = best_feat_xl

# ── Хелпер: gain importance через booster напрямую ───────────────────────────
def get_gain_importance(clf, feat_cols):
    """Всегда возвращает gain — надёжнее чем feature_importances_"""
    raw = clf.booster_.feature_importance(importance_type='gain')
    return pd.DataFrame({
        'feature':    feat_cols,
        'importance': raw
    }).sort_values('importance', ascending=False).reset_index(drop=True)

# ── 1. Подготовка объединённого датасета ─────────────────────────────────────
train_adv = train[FEAT_COLS_ADV].copy(); train_adv['_is_test'] = 0
test_adv  = test[FEAT_COLS_ADV].copy();  test_adv['_is_test']  = 1

df_adv   = pd.concat([train_adv, test_adv], ignore_index=True)
X_adv    = encode_X(df_adv, FEAT_COLS_ADV)
y_adv    = df_adv['_is_test'].values
X_adv_np = X_adv.values if hasattr(X_adv, 'values') else np.array(X_adv)

cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

# ── 2. ROC-AUC curves (5-fold) ───────────────────────────────────────────────
fig, ax = plt.subplots(figsize=(7, 6))
oof_probs = np.zeros(len(X_adv_np))
tprs, aucs_list = [], []
mean_fpr = np.linspace(0, 1, 300)

for fold_i, (tr_idx, val_idx) in enumerate(cv.split(X_adv_np, y_adv)):
    clf_fold = lgb.LGBMClassifier(
        n_estimators=300, num_leaves=63, learning_rate=0.05,
        subsample=0.8, colsample_bytree=0.8,
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    clf_fold.fit(X_adv_np[tr_idx], y_adv[tr_idx])
    raw_probs = clf_fold.predict_proba(X_adv_np[val_idx])[:, 1]
    # Инвертируем если классификатор перепутал классы местами
    probs = raw_probs if np.mean(raw_probs[y_adv[val_idx] == 1]) > 0.5 else 1 - raw_probs
    oof_probs[val_idx] = probs

    fpr, tpr, _ = roc_curve(y_adv[val_idx], probs)
    fold_auc = auc(fpr, tpr)
    aucs_list.append(fold_auc)
    tprs.append(np.interp(mean_fpr, fpr, tpr))
    ax.plot(fpr, tpr, alpha=0.25, linewidth=1, color='#3498db',
            label=f'Fold {fold_i+1} (AUC={fold_auc:.3f})')

mean_tpr = np.mean(tprs, axis=0)
mean_tpr[0] = 0.0; mean_tpr[-1] = 1.0
mean_auc = np.mean(aucs_list)
std_auc  = np.std(aucs_list)
std_tpr  = np.std(tprs, axis=0)

ax.plot(mean_fpr, mean_tpr, color='#e74c3c', linewidth=2.5,
        label=f'Mean ROC (AUC = {mean_auc:.3f} ± {std_auc:.3f})', zorder=5)
ax.fill_between(mean_fpr,
                np.clip(mean_tpr - std_tpr, 0, 1),
                np.clip(mean_tpr + std_tpr, 0, 1),
                alpha=0.15, color='#e74c3c', label='±1 std')
ax.plot([0, 1], [0, 1], linestyle='--', color='gray',
        linewidth=1.5, label='Random (AUC = 0.500)', zorder=1)

if mean_auc < 0.55:
    interp_text = '✅ Нет значимого shift\n(распределения похожи)'
    interp_color = '#27ae60'
elif mean_auc < 0.65:
    interp_text = '⚠️ Умеренный shift\n(стоит изучить топ-фичи)'
    interp_color = '#e67e22'
else:
    interp_text = '❌ Сильный shift\n(модель может деградировать)'
    interp_color = '#c0392b'

ax.text(0.57, 0.12, interp_text, transform=ax.transAxes,
        fontsize=11, color=interp_color, fontweight='bold',
        bbox=dict(boxstyle='round,pad=0.4', facecolor='white',
                  edgecolor=interp_color, alpha=0.9))
ax.set_xlabel('False Positive Rate', fontsize=12)
ax.set_ylabel('True Positive Rate', fontsize=12)
ax.set_title('Adversarial Validation: Train vs Test\n(LightGBM, 5-fold CV)', fontsize=13)
ax.legend(loc='upper left', fontsize=9, framealpha=0.9)
ax.set_xlim([-0.01, 1.01]); ax.set_ylim([-0.01, 1.05])
ax.grid(alpha=0.3)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/adversarial/roc_auc_adversarial.png', dpi=180, bbox_inches='tight')
plt.show()
print(f'✅ ROC saved  |  Mean AUC = {mean_auc:.4f} ± {std_auc:.4f}')

# ── 3. Обучаем clf_adv на ПОЛНОМ датасете — для gain importance и SHAP ────────
clf_adv = lgb.LGBMClassifier(
    n_estimators=300, num_leaves=63, learning_rate=0.05,
    subsample=0.8, colsample_bytree=0.8,
    n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
)
clf_adv.fit(X_adv_np, y_adv)

# Gain через booster — единственный надёжный способ
adv_importance = get_gain_importance(clf_adv, FEAT_COLS_ADV)
print(f'\nTop-15 features by GAIN importance (adversarial):')
print(adv_importance.head(15).to_string(index=False, float_format='{:.2f}'.format))

# ── 4. KS-тест ───────────────────────────────────────────────────────────────
ks_results = []
for col in FEAT_COLS_ADV:
    stat, pval = ks_2samp(train[col].dropna().values, test[col].dropna().values)
    ks_results.append({'feature': col, 'ks_stat': stat, 'ks_pval': pval})
ks_df = pd.DataFrame(ks_results).sort_values('ks_stat', ascending=False).reset_index(drop=True)
ks_df['significant'] = ks_df['ks_pval'] < 0.01

print('\nTop-15 features by KS statistic:')
print(ks_df.head(15).to_string(index=False, float_format='{:.4f}'.format))

# ── 5. Двойная панель: Gain importance + KS ──────────────────────────────────
fig2, axes2 = plt.subplots(1, 2, figsize=(18, 8))

top25_adv = adv_importance.head(25)
axes2[0].barh(top25_adv['feature'][::-1], top25_adv['importance'][::-1],
              color='#e05c5c', alpha=0.85)
axes2[0].axvline(top25_adv['importance'].mean(), color='gray',
                 linestyle='--', alpha=0.5, label='mean')
axes2[0].set_xlabel('Gain Importance (adversarial LGB)')
axes2[0].set_title(f'Adversarial Feature Importance (Gain)\nAUC={mean_auc:.4f}')
axes2[0].legend()

top25_ks = ks_df.head(25)
colors_ks = ['#c0392b' if s else '#5dade2' for s in top25_ks['significant']]
axes2[1].barh(top25_ks['feature'][::-1], top25_ks['ks_stat'][::-1],
              color=colors_ks[::-1], alpha=0.85)
axes2[1].axvline(0.05, color='gray', linestyle='--', alpha=0.5, label='KS=0.05')
axes2[1].set_xlabel('KS Statistic')
axes2[1].set_title('KS Test: train vs test\n(красный = p<0.01)')
axes2[1].legend()

plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/adversarial/adv_importance_ks.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: adv_importance_ks.png')

# ── 6. Ранжируем по gain, строим distribution plots топ-16 ───────────────────
# avg_rank только по gain (как и просили — только gain для значимости)
combined = adv_importance.merge(ks_df, on='feature')
combined['gain_rank'] = combined['importance'].rank(ascending=False)
combined = combined.sort_values('gain_rank').reset_index(drop=True)
TOP_SHIFT = combined.head(16)['feature'].tolist()

fig3, axes3 = plt.subplots(4, 4, figsize=(20, 16))
axes3 = axes3.flatten()

for i, feat in enumerate(TOP_SHIFT):
    ax = axes3[i]
    tr_vals = train[feat].dropna().values
    te_vals = test[feat].dropna().values
    ks_val  = ks_df.loc[ks_df['feature'] == feat, 'ks_stat'].values[0]
    gain_val = adv_importance.loc[adv_importance['feature'] == feat, 'importance'].values[0]

    if pd.Series(tr_vals).nunique() <= 15:
        tr_norm = pd.Series(tr_vals).value_counts(normalize=True).sort_index()
        te_norm = pd.Series(te_vals).value_counts(normalize=True).sort_index()
        idx = sorted(set(tr_norm.index) | set(te_norm.index))
        x = range(len(idx))
        ax.bar([xi - 0.2 for xi in x], [tr_norm.get(v, 0) for v in idx],
               width=0.4, label='train', color='#3498db', alpha=0.7)
        ax.bar([xi + 0.2 for xi in x], [te_norm.get(v, 0) for v in idx],
               width=0.4, label='test', color='#e74c3c', alpha=0.7)
        ax.set_xticks(list(x)); ax.set_xticklabels(idx, rotation=45, fontsize=7)
    else:
        lo = min(np.quantile(tr_vals, 0.01), np.quantile(te_vals, 0.01))
        hi = max(np.quantile(tr_vals, 0.99), np.quantile(te_vals, 0.99))
        xs = np.linspace(lo, hi, 300)
        tr_c = tr_vals[(tr_vals >= lo) & (tr_vals <= hi)]
        te_c = te_vals[(te_vals >= lo) & (te_vals <= hi)]
        try:
            kde_tr = gaussian_kde(tr_c, bw_method=0.3)
            kde_te = gaussian_kde(te_c, bw_method=0.3)
            ax.plot(xs, kde_tr(xs), color='#3498db', linewidth=2, label='train')
            ax.plot(xs, kde_te(xs), color='#e74c3c', linewidth=2,
                    linestyle='--', label='test')
            ax.fill_between(xs, kde_tr(xs), alpha=0.15, color='#3498db')
            ax.fill_between(xs, kde_te(xs), alpha=0.15, color='#e74c3c')
        except Exception:
            ax.hist(tr_c, bins=30, density=True, alpha=0.5, color='#3498db', label='train')
            ax.hist(te_c, bins=30, density=True, alpha=0.5, color='#e74c3c', label='test')

    ax.set_title(f'{feat}\nGain={gain_val:.0f}  KS={ks_val:.3f}', fontsize=9)
    ax.legend(fontsize=7); ax.set_ylabel('')

plt.suptitle('Train vs Test: Top-16 by Gain Importance', fontsize=14, y=1.01)
plt.tight_layout()
plt.savefig(f'{ARTIFACTS_DIR}/adversarial/distribution_shift.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Saved: distribution_shift.png')

# ── 7. По группам communication_type ─────────────────────────────────────────
print('\n─── Adversarial AUC by communication_type (gain-based) ───')
for ct in sorted(train['communication_type'].unique()):
    tr_ct = train[train['communication_type'] == ct][FEAT_COLS_ADV].copy()
    te_ct = test[test['communication_type'] == ct][FEAT_COLS_ADV].copy()
    if len(te_ct) < 100 or len(tr_ct) < 100:
        print(f'  type={ct}: skipped (too few samples)'); continue
    tr_ct['_is_test'] = 0; te_ct['_is_test'] = 1
    df_ct   = pd.concat([tr_ct, te_ct], ignore_index=True)
    X_ct    = encode_X(df_ct, FEAT_COLS_ADV)
    X_ct_np = X_ct.values if hasattr(X_ct, 'values') else np.array(X_ct)
    y_ct    = df_ct['_is_test'].values
    fold_aucs = []
    for tr_i, val_i in StratifiedKFold(n_splits=3, shuffle=True,
                                        random_state=RANDOM_STATE).split(X_ct_np, y_ct):
        clf_ct = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                     n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        clf_ct.fit(X_ct_np[tr_i], y_ct[tr_i])
        raw = clf_ct.predict_proba(X_ct_np[val_i])[:, 1]
        p = raw if np.mean(raw[y_ct[val_i] == 1]) > 0.5 else 1 - raw
        fold_aucs.append(roc_auc_score(y_ct[val_i], p))
    auc_ct = np.mean(fold_aucs)
    flag = '✅' if auc_ct < 0.55 else ('⚠️' if auc_ct < 0.65 else '❌')
    # Gain importance для этого канала
    clf_ct_full = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                      n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    clf_ct_full.fit(X_ct_np, y_ct)
    ct_gain = get_gain_importance(clf_ct_full, FEAT_COLS_ADV)
    top3 = ', '.join(ct_gain.head(3)['feature'].tolist())
    print(f'  type={ct}: AUC={auc_ct:.4f}  {flag}  | top-3 gain: [{top3}]')

# Сохраняем
ks_df.to_csv(f'{ARTIFACTS_DIR}/adversarial/ks_results.csv', index=False)
combined.to_csv(f'{ARTIFACTS_DIR}/adversarial/combined_shift_rank.csv', index=False)
adv_importance.to_csv(f'{ARTIFACTS_DIR}/adversarial/gain_importance.csv', index=False)
print('\n✅ All adversarial artifacts saved')

✅ ROC saved  |  Mean AUC = 0.8878 ± 0.0091

Top-15 features by GAIN importance (adversarial):
           feature  importance
       n_days_life     4183.19
 cus_mark_n_offers     4037.04
          spend_cv     3963.32
              mntv     3951.85
 cat_breadth_ratio     3753.33
       trn_density     3695.88
               age     3528.85
           p_25_tv     3266.28
          n_redeem     3230.35
     cus_cat_5_atv     3189.58
      rto_format_1     3164.72
     mkt_view_rate     3153.92
      rto_format_3     2984.09
cat7_vs_last_visit     2958.56
           p_75_tv     2955.97

Top-15 features by KS statistic:
                 feature  ks_stat  ks_pval  significant
              cat7_share   0.0071   0.0584        False
           cus_cat_7_atv   0.0064   0.1077        False
           cus_cat_6_atv   0.0062   0.1108        False
 cus_cat_days_6_diff_min   0.0060   0.0388        False
           cus_cat_7_std   0.0057   0.6169        False
cus_cat_days_7_diff_mean   0.0055   0.08

KeyboardInterrupt: 

In [55]:
# Проверяем: правда ли что shift в interactions, а не в маргиналях?

# 1. Сравниваем AUC с рандомизированными фичами (permutation test)
from sklearn.metrics import roc_auc_score

# Если AUC на перемешанных данных тоже высокий — значит классификатор 
# учится на структуре данных, а не на реальных фичах
X_perm = X_adv_np.copy()
np.random.shuffle(X_perm)  # перемешиваем строки — уничтожаем correlations
clf_perm = lgb.LGBMClassifier(n_estimators=300, num_leaves=63, learning_rate=0.05,
                               n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
from sklearn.model_selection import cross_val_score
auc_perm = cross_val_score(clf_perm, X_perm, y_adv, cv=3, 
                            scoring='roc_auc').mean()
print(f'AUC на перемешанных данных: {auc_perm:.4f}')
print(f'AUC реальный: {mean_auc:.4f}')
print(f'Разница (реальный shift): {mean_auc - max(auc_perm, 1-auc_perm):.4f}')

# 2. Смотрим train_size vs test_size — размер выборки сам по себе
n_train_adv = (y_adv == 0).sum()
n_test_adv  = (y_adv == 1).sum()
print(f'\nРазмер train: {n_train_adv}, test: {n_test_adv}, ratio: {n_train_adv/n_test_adv:.1f}x')

# 3. Проверяем correlation matrix shift — joint distribution
corr_train = train[FEAT_COLS_ADV].corr()
corr_test  = test[FEAT_COLS_ADV].corr()
corr_diff  = (corr_train - corr_test).abs()
top_corr_pairs = (corr_diff.where(np.triu(np.ones(corr_diff.shape), k=1).astype(bool))
                  .stack().sort_values(ascending=False).head(10))
print('\nТоп-10 пар с наибольшим сдвигом корреляций (joint distribution shift):')
print(top_corr_pairs.to_string())

AUC на перемешанных данных: 0.4989
AUC реальный: 0.8878
Разница (реальный shift): 0.3867

Размер train: 355246, test: 118414, ratio: 3.0x

Топ-10 пар с наибольшим сдвигом корреляций (joint distribution shift):
cus_cat_6_atv  cus_cat_5_std    0.076245
cus_cat_7_std  cus_cat_5_std    0.042301
cus_cat_6_std  cus_cat_5_std    0.033875
cus_cat_6_rto  cus_cat_5_std    0.029322
atv            mntv             0.028965
mtv            mntv             0.028313
p_25_tv        mntv             0.027418
rto_format_2   cus_cat_7_std    0.026367
               cus_cat_6_atv    0.025117
p_75_tv        mntv             0.024599


## 13. Model 8 — Rank-Based Ensemble

Взвешенный ранговый бленд: вес пропорционален OOF uplift@10 каждой базовой модели.

In [28]:
def register_result(name, y_val, scores_val, treat_val, elapsed):
    u10, ci_lo, ci_hi = oof_uplift_at_k(y_val, scores_val, treat_val)
    qauc = oof_qini(y_val, scores_val, treat_val)
    entry = {
        'model':       name,
        'uplift@10':   u10,
        'ci80_lo':     ci_lo,
        'ci80_hi':     ci_hi,
        'qini_auc':    qauc,
        'elapsed_sec': round(elapsed, 1)
    }
    global RESULTS
    # Гарантируем что RESULTS — list
    if not isinstance(RESULTS, list):
        RESULTS = []
    for i, r in enumerate(RESULTS):
        if r['model'] == name:
            RESULTS[i] = entry
            break
    else:
        RESULTS.append(entry)
    print(f'  uplift@10={u10:.4f}  CI80=[{ci_lo:.4f}, {ci_hi:.4f}]  '
          f'qini={qauc:.4f}  ({elapsed:.0f}s)')
    return u10

In [ ]:
print('Building Rank Ensemble...')
t0 = time.time()

from scipy.stats import rankdata

model_scores = {
    'T-Learner':        oof_scores_tl,
    'Hurdle T-Learner': oof_scores_hurdle,
    'X-Learner':        oof_scores_xl,
    'DR-Learner':       oof_scores_dr,
    'Hurdle DR-Learner':oof_scores_hdr,
    'Per-Channel Hurdle':oof_scores_perchan,
    'Optuna Hurdle':    oof_scores_opt,
}

results_lookup = {r['model']: r for r in RESULTS}
weights_raw = {k: max(results_lookup[k]['uplift@10'], 0.01)
               for k in model_scores if k in results_lookup}
w_total  = sum(weights_raw.values())
weights  = {k: v / w_total for k, v in weights_raw.items()}

n = len(train)
blend = np.zeros(n)
for name, scores in model_scores.items():
    w = weights.get(name, 0)
    blend += w * rankdata(scores) / n

elapsed_blend = time.time() - t0
u10_blend = register_result('Rank Ensemble', y, blend, T, elapsed_blend)
np.save(f'{ARTIFACTS_DIR}/oof_blend.npy', blend)

print('\nWeights in ensemble:')
for k, v in sorted(weights.items(), key=lambda x: -x[1]):
    u10 = results_lookup[k]['uplift@10']
    print(f'  {k:30s}: weight={v:.3f}  (uplift@10={u10:.4f})')

Building Rank Ensemble...
  uplift@10=19.1084  CI80=[17.8696, 21.1284]  qini=0.0334  (0s)

Weights in ensemble:
  X-Learner                     : weight=0.172  (uplift@10=19.9622)
  T-Learner                     : weight=0.160  (uplift@10=18.5035)
  Optuna Hurdle                 : weight=0.156  (uplift@10=18.0536)
  DR-Learner                    : weight=0.147  (uplift@10=17.0512)
  Hurdle T-Learner              : weight=0.145  (uplift@10=16.7450)
  Per-Channel Hurdle            : weight=0.130  (uplift@10=15.0752)
  Hurdle DR-Learner             : weight=0.090  (uplift@10=10.3748)


In [ ]:
# ─── MODEL 9: Optuna-tuned X-Learner ─────────────────────────────────────────
print('Training Model 9: Optuna X-Learner...')
t0 = time.time()

# 40% subsample для поиска
from sklearn.model_selection import StratifiedShuffleSplit
sss2 = StratifiedShuffleSplit(n_splits=1, test_size=0.6, random_state=RANDOM_STATE+1)
opt2_idx, _ = next(sss2.split(train, make_stratify_col(train)))
train_opt2  = train.iloc[opt2_idx].reset_index(drop=True)
y_opt2      = train_opt2[TARGET_COL].values
T_opt2      = train_opt2[TREATMENT_COL].values
folds_opt2  = list(StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
                   .split(train_opt2, make_stratify_col(train_opt2)))


def xlearner_oof(params1, params2, prop_model, folds_list, df_sub, y_sub, T_sub, feat_cols):
    """
    X-Learner OOF:
      Stage1: mu1 = f(X|T=1), mu0 = f(X|T=0)
      Stage2: D1 = y1 - mu0(x1), D0 = mu1(x0) - y0  →  tau1, tau0
      Final:  CATE = g(x)*tau0(x) + (1-g(x))*tau1(x)
    """
    from sklearn.linear_model import LogisticRegression
    oof = np.zeros(len(df_sub))

    for _, (tr_idx, val_idx) in enumerate(folds_list):
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]

        mask1 = Ttr == 1; mask0 = Ttr == 0

        # Stage 1
        mu1 = lgb.LGBMRegressor(**params1)
        mu0 = lgb.LGBMRegressor(**params1)
        mu1.fit(X_tr[mask1], ytr[mask1])
        mu0.fit(X_tr[mask0], ytr[mask0])

        # Imputed treatment effects
        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])  # treated: actual - mu0
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]  # control: mu1 - actual

        # Stage 2
        tau1 = lgb.LGBMRegressor(**params2)
        tau0 = lgb.LGBMRegressor(**params2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        # Propensity
        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
            g.fit(X_tr, Ttr)
            g_val = g.predict_proba(X_val)[:, 1]
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
            g.fit(X_tr, Ttr)
            g_val = g.predict_proba(X_val)[:, 1]

        # CATE = g*tau0 + (1-g)*tau1
        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof


def objective_xl(trial):
    # Stage 1 params (outcome models — более сложные)
    params1 = dict(
        n_estimators      = trial.suggest_int('s1_n_est', 100, 500),
        learning_rate     = trial.suggest_float('s1_lr', 0.02, 0.2, log=True),
        num_leaves        = trial.suggest_int('s1_nl', 31, 255),
        min_child_samples = trial.suggest_int('s1_mcs', 10, 80),
        subsample         = trial.suggest_float('s1_ss', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('s1_cst', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 3.0, log=True),
        reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 3.0, log=True),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    # Stage 2 params (CATE models — проще, меньше деревьев)
    params2 = dict(
        n_estimators      = trial.suggest_int('s2_n_est', 50, 300),
        learning_rate     = trial.suggest_float('s2_lr', 0.02, 0.2, log=True),
        num_leaves        = trial.suggest_int('s2_nl', 15, 127),
        min_child_samples = trial.suggest_int('s2_mcs', 10, 80),
        subsample         = trial.suggest_float('s2_ss', 0.6, 1.0),
        colsample_bytree  = trial.suggest_float('s2_cst', 0.5, 1.0),
        reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 3.0, log=True),
        reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 3.0, log=True),
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )
    prop_model = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
    feat_set   = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
    feat_cols  = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP

    try:
        score, _ = xlearner_oof(params1, params2, prop_model,
                                folds_opt2, train_opt2, y_opt2, T_opt2, feat_cols)
        return score
    except Exception as e:
        return -999.0


study_xl = optuna.create_study(
    direction='maximize',
    sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
    pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)
)
# timeout=2700 = 45 минут на поиск, ~15 минут остаётся на финальный OOF
study_xl.optimize(objective_xl, n_trials=200, timeout=2700, show_progress_bar=True)

best_xl = study_xl.best_params
print(f'\nBest Optuna XL trial: uplift@10={study_xl.best_value:.4f}  '
      f'trials_done={len(study_xl.trials)}')
print(f'prop_model={best_xl["prop_model"]},  feat_set={best_xl["feat_set"]}')

# Масштабируем n_estimators для финального обучения
def scale_params(p, prefix, scale=2.5):
    return dict(
        n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 1500),
        learning_rate     = p[f'{prefix}_lr'],
        num_leaves        = p[f'{prefix}_nl'],
        min_child_samples = p[f'{prefix}_mcs'],
        subsample         = p[f'{prefix}_ss'],
        colsample_bytree  = p[f'{prefix}_cst'],
        reg_alpha         = p[f'{prefix}_ra'],
        reg_lambda        = p[f'{prefix}_rl'],
        n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
    )

best_params1_xl = scale_params(best_xl, 's1')
best_params2_xl = scale_params(best_xl, 's2')
best_prop_xl    = best_xl['prop_model']
best_feat_xl    = FEAT_COLS if best_xl['feat_set'] == 'all' else FEAT_COLS_TOP

# Финальный OOF на полных данных (5-fold)
print('\nRunning final 5-fold OOF on full train...')
_, oof_scores_xl_opt = xlearner_oof(
    best_params1_xl, best_params2_xl, best_prop_xl,
    folds, train, y, T, best_feat_xl
)

elapsed_xl = time.time() - t0
u10_xl = register_result('Optuna X-Learner', y, oof_scores_xl_opt, T, elapsed_xl)
np.save(f'{ARTIFACTS_DIR}/oof_xlearner_opt.npy', oof_scores_xl_opt)
with open(f'{ARTIFACTS_DIR}/optuna_xl_params.json', 'w') as f:
    json.dump({
        'params1': best_params1_xl,
        'params2': best_params2_xl,
        'prop_model': best_prop_xl,
        'feat_set': best_xl['feat_set'],
        'search_uplift10': study_xl.best_value,
        'trials_completed': len(study_xl.trials)
    }, f, indent=2)

Training Model 9: Optuna X-Learner...


In [9]:
# ══════════════════════════════════════════════════════════════
#  ДИАГНОСТИКА ЛУЧШЕЙ МОДЕЛИ — вставь и запусти
# ══════════════════════════════════════════════════════════════
import json, shap, matplotlib.pyplot as plt
from scipy.stats import spearmanr

# ── Загружаем лучшие параметры ────────────────────────────────
with open(f'{ARTIFACTS_DIR}/optuna_xl_params.json') as f:
    xl_cfg = json.load(f)

params1_diag = xl_cfg['params1']
params2_diag = xl_cfg['params2']
feat_cols_diag = FEAT_COLS  # feat_set='all'

# ── 1. Stability: OOF uplift@10 на 5 разных random_state ─────
print('=== 1. STABILITY CHECK ===')
seeds = [42, 123, 777, 1337, 2025]
stab_results = []
for seed in seeds:
    folds_s = list(StratifiedKFold(n_splits=5, shuffle=True, random_state=seed)
                   .split(train, make_stratify_col(train)))
    score_s, _ = xlearner_oof(params1_diag, params2_diag, 'logreg',
                               folds_s, train, y, T, feat_cols_diag)
    stab_results.append(score_s)
    print(f'  seed={seed}: uplift@10={score_s:.4f}')
print(f'  mean={np.mean(stab_results):.4f}  std={np.std(stab_results):.4f}  '
      f'min={np.min(stab_results):.4f}  max={np.max(stab_results):.4f}')

# ── 2. Uplift@k curve — смотрим не только @10 ────────────────
print('\n=== 2. UPLIFT@K CURVE ===')
oof_best = np.load(f'{ARTIFACTS_DIR}/oof_xlearner_opt.npy')
for k in [0.05, 0.10, 0.15, 0.20, 0.30, 0.50]:
    print(f'  uplift@{int(k*100):02d}% = {uplift_at_k_spend(y, oof_best, T, k=k):.4f}')

# ── 3. Score distribution — outliers, saturated tails ────────
print('\n=== 3. SCORE DISTRIBUTION ===')
print(f'  min={oof_best.min():.4f}  p1={np.percentile(oof_best,1):.4f}  '
      f'p5={np.percentile(oof_best,5):.4f}  p25={np.percentile(oof_best,25):.4f}')
print(f'  median={np.median(oof_best):.4f}  mean={oof_best.mean():.4f}')
print(f'  p75={np.percentile(oof_best,75):.4f}  p95={np.percentile(oof_best,95):.4f}  '
      f'p99={np.percentile(oof_best,99):.4f}  max={oof_best.max():.4f}')
print(f'  % negative scores: {(oof_best < 0).mean()*100:.1f}%')
print(f'  % score > 100:     {(oof_best > 100).mean()*100:.1f}%')

# ── 4. Score vs target correlation по decilям ─────────────────
print('\n=== 4. DECILE TABLE (OOF scores vs actual uplift) ===')
df_dec = pd.DataFrame({'score': oof_best, 'y': y, 'T': T})
df_dec['decile'] = pd.qcut(df_dec['score'].rank(method='first'), 10,
                            labels=[f'D{i}' for i in range(1, 11)])
dec_tbl = df_dec.groupby('decile', observed=True).apply(
    lambda g: pd.Series({
        'n': len(g),
        'mean_score': g['score'].mean(),
        'uplift': (g.loc[g.T==1,'y'].mean() - g.loc[g.T==0,'y'].mean())
                   if g['T'].nunique()==2 else np.nan,
        'n_treated': (g['T']==1).sum(),
        'n_control': (g['T']==0).sum(),
    })
).reset_index()
print(dec_tbl.to_string(index=False))

# ── 5. Per-channel performance ────────────────────────────────
print('\n=== 5. PER COMMUNICATION_TYPE PERFORMANCE ===')
for ch in sorted(train['communication_type'].unique()):
    mask = train['communication_type'] == ch
    if mask.sum() < 100: continue
    ch_score = uplift_at_k_spend(y[mask], oof_best[mask], T[mask], k=0.1)
    n_t = T[mask].sum(); n_c = (1-T[mask]).sum()
    print(f'  {ch}: uplift@10={ch_score:.4f}  n_treat={n_t}  n_ctrl={n_c}')

# ── 6. SHAP — топ-20 фичей Stage 1 ───────────────────────────
print('\n=== 6. SHAP FEATURE IMPORTANCE (Stage 1 mu0) ===')
X_sample = encode_X(train.sample(5000, random_state=42), feat_cols_diag)
mask0_full = T == 0
mu0_diag = lgb.LGBMRegressor(**params1_diag)
mu0_diag.fit(encode_X(train[mask0_full], feat_cols_diag), y[mask0_full])
explainer = shap.TreeExplainer(mu0_diag)
shap_vals = explainer.shap_values(X_sample)
shap_imp  = pd.DataFrame({
    'feature': feat_cols_diag[:X_sample.shape[1]],  # осторожно с encode
    'mean_abs_shap': np.abs(shap_vals).mean(axis=0)
}).sort_values('mean_abs_shap', ascending=False).head(25)
print(shap_imp.to_string(index=False))

# ── 7. D1/D0 distribution — качество псевдо-эффектов ─────────
print('\n=== 7. PSEUDO-EFFECT (D1/D0) DISTRIBUTION ===')
# Переобучаем Stage1 на всём трейне для диагностики
mu1_diag = lgb.LGBMRegressor(**params1_diag)
mu0_diag2 = lgb.LGBMRegressor(**params1_diag)
X_full = encode_X(train, feat_cols_diag)
mu1_diag.fit(X_full[T==1], y[T==1])
mu0_diag2.fit(X_full[T==0], y[T==0])
D1_diag = y[T==1] - mu0_diag2.predict(X_full[T==1])
D0_diag = mu1_diag.predict(X_full[T==0]) - y[T==0]
for name, arr in [('D1 (treated)', D1_diag), ('D0 (control)', D0_diag)]:
    print(f'  {name}: mean={arr.mean():.3f}  std={arr.std():.3f}  '
          f'% positive={( arr>0).mean()*100:.1f}%  '
          f'p5={np.percentile(arr,5):.2f}  p95={np.percentile(arr,95):.2f}')

# ── 8. Spearman correlation OOF score vs D1/D0 ───────────────
print('\n=== 8. OOF SCORE CORRELATION WITH D1/D0 ===')
# Для treated клиентов
idx_t = np.where(T==1)[0]
rho_t, _ = spearmanr(oof_best[idx_t], D1_diag)
idx_c = np.where(T==0)[0]
rho_c, _ = spearmanr(oof_best[idx_c], D0_diag)
print(f'  Spearman(score_treated, D1) = {rho_t:.4f}')
print(f'  Spearman(score_control, D0) = {rho_c:.4f}')
print('  (если < 0.3 — Stage2 плохо учит псевдо-эффекты)')

print('\n✅ Диагностика завершена — пришли вывод!')

=== 1. STABILITY CHECK ===


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

  seed=42: uplift@10=21.0739


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://sciki

KeyboardInterrupt: 

In [ ]:
# ═══ MODEL 12: Per-Channel Optuna X-Learner ══════════════════════════════════
print('Training Model 12: Per-Channel Optuna X-Learner...')
t0 = time.time()

COMM_CODES = sorted(train[COMM_COL].unique())
print(f'Channels: {COMM_CODES}')
# Проверим размеры
for ch in COMM_CODES:
    mask = train[COMM_COL] == ch
    n_t  = T[mask].sum()
    n_c  = (1 - T[mask]).sum()
    print(f'  {ch}: n={mask.sum():,}  treated={n_t:,}  control={n_c:,}')

# ── Функция X-Learner OOF для произвольного подмножества ─────────────────────
def xlearner_oof_subset(params1, params2, prop_model, folds_list,
                        df_sub, y_sub, T_sub, feat_cols):
    """X-Learner OOF на произвольном подмножестве данных."""
    oof = np.zeros(len(df_sub))
    for tr_idx, val_idx in folds_list:
        X_tr  = encode_X(df_sub.iloc[tr_idx], feat_cols)
        X_val = encode_X(df_sub.iloc[val_idx], feat_cols)
        ytr   = y_sub[tr_idx]
        Ttr   = T_sub[tr_idx]
        mask1 = Ttr == 1; mask0 = Ttr == 0

        mu1 = lgb.LGBMRegressor(**params1); mu0 = lgb.LGBMRegressor(**params1)
        mu1.fit(X_tr[mask1], ytr[mask1])
        mu0.fit(X_tr[mask0], ytr[mask0])

        D1 = ytr[mask1] - mu0.predict(X_tr[mask1])
        D0 = mu1.predict(X_tr[mask0]) - ytr[mask0]

        tau1 = lgb.LGBMRegressor(**params2); tau0 = lgb.LGBMRegressor(**params2)
        tau1.fit(X_tr[mask1], D1)
        tau0.fit(X_tr[mask0], D0)

        if prop_model == 'logreg':
            g = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
        else:
            g = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                   n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
        g.fit(X_tr, Ttr)
        g_val = g.predict_proba(X_val)[:, 1]

        oof[val_idx] = g_val * tau0.predict(X_val) + (1 - g_val) * tau1.predict(X_val)

    return uplift_at_k_spend(y_sub, oof, T_sub, k=0.1), oof


# ── Per-channel Optuna + финальный OOF ───────────────────────────────────────
channel_params  = {}   # лучшие параметры по каналу
channel_studies = {}
oof_perchan_xl  = np.zeros(len(train))
test_scores_perchan_xl = np.zeros(len(test))

for ch in COMM_CODES:
    print(f'\n{"="*60}')
    print(f'  Channel: {ch}')
    t_ch = time.time()

    # Маски для трейна
    ch_mask_train = (train[COMM_COL] == ch).values
    df_ch   = train[ch_mask_train].reset_index(drop=True)
    y_ch    = y[ch_mask_train]
    T_ch    = T[ch_mask_train]

    # Маска для теста
    ch_mask_test = (test[COMM_COL] == ch).values
    df_ch_test   = test[ch_mask_test].reset_index(drop=True)

    n_ch = len(df_ch)
    print(f'  Train: {n_ch:,}  treated={T_ch.sum():,}  control={(1-T_ch).sum():,}')
    print(f'  Test:  {ch_mask_test.sum():,}')

    # ── 40% subsample для Optuna ─────────────────────────────────────────────
    sss_ch = StratifiedShuffleSplit(n_splits=1, test_size=0.6,
                                    random_state=RANDOM_STATE + hash(ch) % 100)
    opt_idx_ch, _ = next(sss_ch.split(df_ch, make_stratify_col(df_ch)))
    df_opt_ch  = df_ch.iloc[opt_idx_ch].reset_index(drop=True)
    y_opt_ch   = y_ch[opt_idx_ch]
    T_opt_ch   = T_ch[opt_idx_ch]
    folds_opt_ch = list(
        StratifiedKFold(n_splits=2, shuffle=True, random_state=RANDOM_STATE)
        .split(df_opt_ch, make_stratify_col(df_opt_ch))
    )

    def make_objective_ch(df_o, y_o, T_o, f_opt_ch):
        """Замыкание: objective для конкретного канала."""
        def objective_ch(trial):
            params1 = dict(
                n_estimators      = trial.suggest_int('s1_n_est', 100, 500),
                learning_rate     = trial.suggest_float('s1_lr', 0.02, 0.2, log=True),
                num_leaves        = trial.suggest_int('s1_nl', 31, 255),
                min_child_samples = trial.suggest_int('s1_mcs', 10, 80),
                subsample         = trial.suggest_float('s1_ss', 0.6, 1.0),
                colsample_bytree  = trial.suggest_float('s1_cst', 0.5, 1.0),
                reg_alpha         = trial.suggest_float('s1_ra', 1e-3, 3.0, log=True),
                reg_lambda        = trial.suggest_float('s1_rl', 1e-3, 3.0, log=True),
                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
            )
            params2 = dict(
                n_estimators      = trial.suggest_int('s2_n_est', 50, 300),
                learning_rate     = trial.suggest_float('s2_lr', 0.02, 0.2, log=True),
                num_leaves        = trial.suggest_int('s2_nl', 15, 127),
                min_child_samples = trial.suggest_int('s2_mcs', 10, 80),
                subsample         = trial.suggest_float('s2_ss', 0.6, 1.0),
                colsample_bytree  = trial.suggest_float('s2_cst', 0.5, 1.0),
                reg_alpha         = trial.suggest_float('s2_ra', 1e-3, 3.0, log=True),
                reg_lambda        = trial.suggest_float('s2_rl', 1e-3, 3.0, log=True),
                n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
            )
            prop_model = trial.suggest_categorical('prop_model', ['logreg', 'lgb'])
            feat_set   = trial.suggest_categorical('feat_set', ['all', 'top25_ext'])
            feat_cols  = FEAT_COLS if feat_set == 'all' else FEAT_COLS_TOP
            try:
                score, _ = xlearner_oof_subset(
                    params1, params2, prop_model,
                    f_opt_ch, df_o, y_o, T_o, feat_cols
                )
                return score
            except Exception as e:
                return -999.0
        return objective_ch

    # ── Optuna для канала ────────────────────────────────────────────────────
    study_ch = optuna.create_study(
        direction='maximize',
        sampler=optuna.samplers.TPESampler(seed=RANDOM_STATE),
        pruner=optuna.pruners.MedianPruner(n_startup_trials=5, n_warmup_steps=0)
    )
    # ~15 мин на канал (3 канала = ~45 мин)
    study_ch.optimize(
        make_objective_ch(df_opt_ch, y_opt_ch, T_opt_ch, folds_opt_ch),
        n_trials=60, timeout=900, show_progress_bar=True
    )
    best_ch = study_ch.best_params
    channel_studies[ch] = study_ch
    print(f'  Best trial uplift@10={study_ch.best_value:.4f}  '
          f'trials={len(study_ch.trials)}')
    print(f'  prop={best_ch["prop_model"]}  feat={best_ch["feat_set"]}')

    # ── Масштабируем n_estimators для финального обучения ────────────────────
    def scale_ch(p, prefix, scale=2.5):
        return dict(
            n_estimators      = min(int(p[f'{prefix}_n_est'] * scale), 1500),
            learning_rate     = p[f'{prefix}_lr'],
            num_leaves        = p[f'{prefix}_nl'],
            min_child_samples = p[f'{prefix}_mcs'],
            subsample         = p[f'{prefix}_ss'],
            colsample_bytree  = p[f'{prefix}_cst'],
            reg_alpha         = p[f'{prefix}_ra'],
            reg_lambda        = p[f'{prefix}_rl'],
            n_jobs=-1, random_state=RANDOM_STATE, verbose=-1
        )

    p1_ch   = scale_ch(best_ch, 's1')
    p2_ch   = scale_ch(best_ch, 's2')
    prop_ch = best_ch['prop_model']
    feat_ch = FEAT_COLS if best_ch['feat_set'] == 'all' else FEAT_COLS_TOP
    channel_params[ch] = {'params1': p1_ch, 'params2': p2_ch,
                          'prop': prop_ch, 'feat': best_ch['feat_set']}

    # ── Финальный OOF на канале (5-fold) ─────────────────────────────────────
    folds_ch_full = list(
        StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)
        .split(df_ch, make_stratify_col(df_ch))
    )
    u10_ch, oof_ch = xlearner_oof_subset(
        p1_ch, p2_ch, prop_ch,
        folds_ch_full, df_ch, y_ch, T_ch, feat_ch
    )
    oof_perchan_xl[ch_mask_train] = oof_ch
    print(f'  Final OOF uplift@10={u10_ch:.4f}  ({time.time()-t_ch:.0f}s)')

    # ── Предсказания на тесте для канала ─────────────────────────────────────
    X_tr_full = encode_X(df_ch, feat_ch)
    X_te_ch   = encode_X(df_ch_test, feat_ch)
    mask1_f   = T_ch == 1; mask0_f = T_ch == 0

    mu1_f = lgb.LGBMRegressor(**p1_ch); mu0_f = lgb.LGBMRegressor(**p1_ch)
    mu1_f.fit(X_tr_full[mask1_f], y_ch[mask1_f])
    mu0_f.fit(X_tr_full[mask0_f], y_ch[mask0_f])

    D1_f = y_ch[mask1_f] - mu0_f.predict(X_tr_full[mask1_f])
    D0_f = mu1_f.predict(X_tr_full[mask0_f]) - y_ch[mask0_f]

    tau1_f = lgb.LGBMRegressor(**p2_ch); tau0_f = lgb.LGBMRegressor(**p2_ch)
    tau1_f.fit(X_tr_full[mask1_f], D1_f)
    tau0_f.fit(X_tr_full[mask0_f], D0_f)

    if prop_ch == 'logreg':
        g_f = LogisticRegression(max_iter=300, C=1.0, n_jobs=-1)
    else:
        g_f = lgb.LGBMClassifier(n_estimators=200, num_leaves=31,
                                  n_jobs=-1, random_state=RANDOM_STATE, verbose=-1)
    g_f.fit(X_tr_full, T_ch)
    g_te = g_f.predict_proba(X_te_ch)[:, 1]

    test_scores_perchan_xl[ch_mask_test] = (
        g_te * tau0_f.predict(X_te_ch) + (1 - g_te) * tau1_f.predict(X_te_ch)
    )

# ── Итоговые метрики ──────────────────────────────────────────────────────────
elapsed_pcxl = time.time() - t0
u10_pcxl = uplift_at_k_spend(y, oof_perchan_xl, T, k=0.1)
u20_pcxl = uplift_at_k_spend(y, oof_perchan_xl, T, k=0.2)
print(f'\n{"="*60}')
print(f'Per-Channel Optuna X-Learner:')
print(f'  uplift@10={u10_pcxl:.4f}  uplift@20={u20_pcxl:.4f}')
print(f'  elapsed={elapsed_pcxl:.0f}s')

# Per-channel breakdown
print('\nPer-channel OOF uplift@10:')
for ch in COMM_CODES:
    mask = (train[COMM_COL] == ch).values
    u_ch = uplift_at_k_spend(y[mask], oof_perchan_xl[mask], T[mask], k=0.1)
    print(f'  {ch}: {u_ch:.4f}')

np.save(f'{ARTIFACTS_DIR}/oof_perchan_xl_opt.npy', oof_perchan_xl)

# ── Сабмит ────────────────────────────────────────────────────────────────────
submission_pcxl = pd.DataFrame({
    'user_id':      test['user_id'],
    'UPLIFT_SCORE': test_scores_perchan_xl
})
sub_path_pcxl = f'{ARTIFACTS_DIR}/submission_perchan_xl_opt.csv'
submission_pcxl.to_csv(sub_path_pcxl, index=False)
print(f'\n✅ Saved: {sub_path_pcxl}')
print(f'   score stats: min={test_scores_perchan_xl.min():.3f}  '
      f'max={test_scores_perchan_xl.max():.3f}  '
      f'mean={test_scores_perchan_xl.mean():.3f}')

# Сохраняем параметры по каналам
with open(f'{ARTIFACTS_DIR}/perchan_xl_params.json', 'w') as f:
    json.dump({str(ch): {k: v for k, v in d.items() if k != 'feat'}
               for ch, d in channel_params.items()}, f, indent=2)

# ── Бленд с глобальной моделью ────────────────────────────────────────────────
print('\n--- Blend: per-channel XL vs global XL ---')
oof_global = np.load(f'{ARTIFACTS_DIR}/oof_xlearner_opt.npy')
from scipy.stats import rankdata

for w in [0.3, 0.4, 0.5, 0.6, 0.7]:
    r1 = rankdata(oof_perchan_xl)
    r2 = rankdata(oof_global)
    blend = w * r1 + (1 - w) * r2
    u10_b = uplift_at_k_spend(y, blend, T, k=0.1)
    print(f'  perchan*{w:.1f} + global*{1-w:.1f}: uplift@10={u10_b:.4f}')

Training Model 12: Per-Channel Optuna X-Learner...
Channels: [np.int64(0), np.int64(1), np.int64(2)]
  0: n=118,309  treated=58,589  control=59,720
  1: n=118,466  treated=58,833  control=59,633
  2: n=118,471  treated=58,610  control=59,861

  Channel: 0
  Train: 118,309  treated=58,589  control=59,720
  Test:  39,436


Best trial: 0. Best value: 16.217:   2%|▏         | 1/60 [00:54<53:07, 54.02s/it, 54.02/900 seconds]/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preproc

  Best trial uplift@10=23.6773  trials=28
  prop=logreg  feat=all


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Final OOF uplift@10=21.3902  (1031s)

  Channel: 1
  Train: 118,466  treated=58,833  control=59,633
  Test:  39,488


...

Best trial: 0. Best value: 28.0275:   2%|▏         | 1/60 [00:44<43:58, 44.72s/it, 44.72/900 seconds]/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(
/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/prepro

  Best trial uplift@10=35.0482  trials=26
  prop=logreg  feat=all


/Users/olegandreev/Downloads/Кейс 2. Uplift/.venv/lib/python3.14/site-packages/sklearn/linear_model/_logistic.py:465: ConvergenceWarning: lbfgs failed to converge (status=1):
STOP: TOTAL NO. OF ITERATIONS REACHED LIMIT.

Increase the number of iterations (max_iter) or scale the data as shown in:
    https://scikit-learn.org/stable/modules/preprocessing.html
Please also refer to the documentation for alternative solver options:
    https://scikit-learn.org/stable/modules/linear_model.html#logistic-regression
  n_iter_i = _check_optimize_result(


  Final OOF uplift@10=28.6966  (1053s)

Per-Channel Optuna X-Learner:
  uplift@10=20.6058  uplift@20=11.5745
  elapsed=3693s

Per-channel OOF uplift@10:
  0: 21.3902
  1: 9.1778
  2: 28.6966

✅ Saved: pilot_artifacts/submission_perchan_xl_opt.csv
   score stats: min=-177.765  max=393.379  mean=3.251


TypeError: keys must be str, int, float, bool or None, not int64